In [9]:
print("[fig04b] Physics coupling …")
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
from scipy.stats import pearsonr
from matplotlib.transforms import blended_transform_factory

def get_real_feat_name(hints, columns):
    for hint in hints:
        for col in columns:
            if hint.lower().replace(" ", "") in col.lower().replace(" ", ""):
                return col
    return hints[0]

PHYS_PAIRS_HINTS = [
    (["Nighttime Temp (10th Percentile)", "LST_Night_1km_p10"], 
     "Nighttime LST (10th Pct, °C)", C["red"]),
    (["Thermal Vegetation Index (TVI)", "ENG_TVI"], 
     "Thermal Vegetation Index (TVI)", C["green"]),
    (["Daytime Temp (10th Percentile)", "LST_Day_1km_p10"], 
     "Daytime LST (10th Pct, °C)", C["orange"]),
]

fig4b = plt.figure(figsize=(18, 15)) 
fig4b.suptitle(
    "Physics of Life Expectancy: Raw Environmental Signal vs. Model Attribution\n"
    "Left: raw data density — Right: SHAP-attributed effect (years)",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.04) 
gs4b = gridspec.GridSpec(3, 2, figure=fig4b, hspace=0.52, wspace=0.34)
letters = ["(A)","(B)","(C)","(D)","(E)","(F)"]
pi = 0

for row_i, (hints, disp_name, col) in enumerate(PHYS_PAIRS_HINTS):
    ax_r  = fig4b.add_subplot(gs4b[row_i, 0])
    ax_sh = fig4b.add_subplot(gs4b[row_i, 1])
    
    raw_feat = get_real_feat_name(hints, df_full.columns)
    shap_feat = get_real_feat_name(hints, fn_list)
    is_veg = any(v in raw_feat.lower() for v in ["ndvi", "tvi", "vegetation", "evi", "savi"])

    # --- UNIFIED CLEAN TITLE ---
    # This forces the left panel to use the beautiful SHAP name instead of the raw CSV header
    clean_title = shap_feat.replace("_", " ")

    # ==========================================
    # LEFT PANEL: RAW HEXBIN
    # ==========================================
    if raw_feat in df_full.columns and LE_COL in df_full.columns:
        tmp = df_full[[raw_feat, LE_COL]].dropna()
        xr  = np.asarray(tmp[raw_feat].values)
        if is_veg:
            veg_sc = 10000.0 if xr.max() > 10 else 1.0
            xr = xr / veg_sc
            
        xn  = normalize01(xr)
        yr_ = tmp[LE_COL].values
        
        hb  = ax_r.hexbin(xn, yr_, gridsize=42, cmap="viridis", mincnt=1, alpha=0.92, linewidths=0.15)
        cb  = fig4b.colorbar(hb, ax=ax_r, shrink=0.75, pad=0.02)
        cb.set_label("County-year density", fontsize=FS_ANNOT)
        cb.ax.tick_params(labelsize=FS_ANNOT)
        
        try:
            ord_ = np.argsort(xn)
            lw_  = sm_lowess(yr_[ord_], xn[ord_], frac=0.25, return_sorted=True)
            ax_r.plot(lw_[:,0], lw_[:,1], color="white", lw=2.8, ls="--", zorder=5)
            ax_r.plot(lw_[:,0], lw_[:,1], color=col, lw=1.8, ls="--", zorder=6, alpha=0.85, label="LOWESS trend")
        except Exception:
            pass
            
        rv2, pv2 = pearsonr(xn, yr_)
        xlab = ("Index Value (0=low, 1=high)" if is_veg else f"{disp_name} (normalized 0–1)")
        
        ax_r.set_xlabel(xlab, fontweight="bold", fontsize=FS_LABEL)
        ax_r.set_ylabel("Life expectancy  (years)", fontweight="bold", fontsize=FS_LABEL)
        ax_r.set_xlim(-0.02, 1.02)
        ax_r.grid(True, alpha=0.18)
        
        # APPLYING CLEAN TITLE HERE
        ax_r.set_title(f"{letters[pi]}  {clean_title}", fontweight="bold", loc="left", fontsize=FS_TITLE, pad=12)
                       
        stat_box(ax_r, f"Pearson r = {rv2:.3f}  (p<0.001)\n" f"n = {len(tmp):,} county-years",
                 loc="lower right", fs=FS_ANNOT, fc="#FFFDE7" if rv2 < 0 else "white")
        ax_r.legend(fontsize=FS_LEGEND-1, loc="upper right", framealpha=0.92)
    
    pi += 1

    # ==========================================
    # RIGHT PANEL: SHAP ATTRIBUTION
    # ==========================================
    if shap_feat in fn_list:
        sidx  = fn_list.index(shap_feat)
        xs    = np.asarray(X_sample[shap_feat].values.copy())
        ys    = np.asarray(shap_values_arr[:, sidx])
        
        if is_veg and xs.max() > 10:
            xs = xs / 10000.0
            
        sc_ = ax_sh.scatter(xs, ys, c=xs, cmap="RdYlGn" if is_veg else "coolwarm", alpha=0.45, s=14, linewidths=0, rasterized=True)
        cb2 = fig4b.colorbar(sc_, ax=ax_sh, shrink=0.75, pad=0.02)
        cb2.set_label("Feature value", fontsize=FS_ANNOT)
        cb2.ax.tick_params(labelsize=FS_ANNOT)
        
        try:
            lx2, ly2 = lowess_trend(xs, ys, frac=0.20)
            ax_sh.plot(lx2, ly2, color=col, lw=2.8, zorder=5, label="LOWESS")
            sc_idx = np.where(np.diff(np.sign(ly2)))[0]
            thresholds = [float(lx2[sci]) for sci in sc_idx[:2]] 
            unit = "" if is_veg else " °C"
            
            if len(thresholds) > 0:
                for xc in thresholds:
                    ax_sh.axvline(xc, color="#333", lw=1.8, ls=":", alpha=0.8)
                
                thr_str = " & ".join([f"{x:.2f}{unit}" for x in thresholds])
                lbl = f"Threshold\n@ {thr_str}" if len(thresholds) == 1 else f"Thresholds\n@ {thr_str}"
                
                ann_x = thresholds[0] + (xs.max() - xs.min()) * 0.04 if len(thresholds) == 1 else np.mean(thresholds)
                ha_ = "left" if len(thresholds) == 1 else "center"
                
                curve_mid_y = np.median(ly2)
                ann_y_axis_coords = 0.75 if curve_mid_y < 0 else 0.25 
                trans_popout = blended_transform_factory(ax_sh.transData, ax_sh.transAxes)
                
                ax_sh.text(ann_x, ann_y_axis_coords, lbl, transform=trans_popout, fontsize=FS_ANNOT, color="#333", ha=ha_, va="center",
                           bbox=dict(boxstyle="round,pad=0.45", fc="white", alpha=0.85, ec=C["grey"], lw=1.5, zorder=12))
        except Exception:
            pass
            
        ax_sh.axhline(0, color="#666", lw=1.0, ls="--", alpha=0.6, label="Zero SHAP")
        xlab2 = ("Index Value (0=bare, 1=dense veg)" if is_veg else f"{disp_name} (raw °C)")
        
        ax_sh.set_xlabel(xlab2, fontweight="bold", fontsize=FS_LABEL)
        ax_sh.set_ylabel("SHAP value  (Δ LE, years)", fontweight="bold", fontsize=FS_LABEL)
        ax_sh.grid(True, alpha=0.18)
        
        # APPLYING CLEAN TITLE HERE
        ax_sh.set_title(f"{letters[pi]}  SHAP attribution: {clean_title}", fontweight="bold", loc="left", fontsize=FS_TITLE, pad=12)
                        
        stat_box(ax_sh, f"Mean |SHAP| = {float(np.mean(np.abs(ys))):.3f} yr", loc="upper right", fs=FS_ANNOT)
        ax_sh.legend(fontsize=FS_LEGEND-1, framealpha=0.92, loc="lower left")
        
    pi += 1

fig4b.text(0.5, 0.02,
           "Physics interpretation: nighttime LST = chronic radiative heat load; "
           "TVI = vegetation-albedo-cooling feedback; "
           "daytime LST = peak energy-balance stress.\n"
           "SHAP values confirm causal dominance after controlling for other features.",
           ha="center", fontsize=FS_ANNOT, style="italic", color="#444")

save_fig(fig4b, "fig04b_physics_coupling_v5")

[fig04b] Physics coupling …
  ✓  fig04b_physics_coupling_v5.pdf / .png


In [ ]:

# ================================================================================
# Plotting codes for paper figures (v5) with consistent style, fixed text overlaps, restructured panels
# ================================================================================

"""
================================================================================
MULTIMODAL LIFE EXPECTANCY PREDICTION — PUBLICATION FIGURES (v5)
All figures with consistent style, fixed text overlaps, restructured panels
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import cm, colors as mcolors, lines as mlines
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MultipleLocator, MaxNLocator
from matplotlib.patches import Rectangle
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
import shap
import geopandas as gpd
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from matplotlib.patches import Rectangle
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

# ── Output directory ─────────────────────────────────────────────────────────
OUT = Path('/Users/faizahmad/Desktop/paper1/paper1_run1/figures')
OUT.mkdir(parents=True, exist_ok=True)

# ── File paths ────────────────────────────────────────────────────────────────
RESULTS_FILE = '/Users/faizahmad/Desktop/paper1/paper1_run1/results_production_final/ml_results.pkl'
DATA_FILE    = '/Users/faizahmad/Desktop/paper1/paper1_run1/full_clean_engineered_dataset_with_LE.csv'
COUNTY_SHP   = Path("/Users/faizahmad/Desktop/grok_live/data/cb_2018_us_county_500k/cb_2018_us_county_500k.shp")
STATE_SHP    = Path("/Users/faizahmad/Desktop/grok_live/data/cb_2018_us_state_500k/cb_2018_us_state_500k.shp")

# ─────────────────────────────────────────────────────────────────────────────
# 1. GLOBAL STYLE CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
FONT_FAMILY   = ["Helvetica Neue", "Arial", "DejaVu Sans"]
FS_BASE       = 14
FS_LABEL      = 18
FS_TITLE      = 16
FS_SUPTITLE   = 20
FS_PANEL_TAG  = 16
FS_ANNOT      = 13
FS_LEGEND     = 13

DPI_SCREEN    = 150
DPI_SAVE      = 300
PAD           = 0.18   # slightly more generous than v3

C = dict(
    blue    = "#0072B2",
    orange  = "#E69F00",
    green   = "#009E73",
    red     = "#D55E00",
    purple  = "#CC79A7",
    sky     = "#56B4E9",
    yellow  = "#F0E442",
    black   = "#111111",
    grey    = "#AAAAAA",
    night   = "#1A1A4E",
    day     = "#FF8C00",
    lgrey   = "#EEEEEE",
)

CMAP_DIV  = "RdBu_r"
CMAP_ERR  = "YlOrRd"
CMAP_LE   = "RdYlGn"

CONTINENTAL = [str(i).zfill(2) for i in range(1, 57) if i not in [2, 15, 72]]

CAT_PAL = {
    "Temperature":  C["red"],
    "Livestock":    C["orange"],
    "Development":  C["purple"],
    "Vegetation":   C["green"],
    "Topography":   C["sky"],
    "Water":        C["blue"],
    "Soil":         "#8C4A2F",
    "Agriculture":  "#999900",
    "SAR":          C["black"],
    "Other":        C["grey"],
}

plt.rcParams.update({
    "font.family":           "sans-serif",
    "font.sans-serif":       FONT_FAMILY,
    "font.size":             FS_SUPTITLE,  # default font size (can be overridden by specific elements)
    "axes.titlesize":        FS_TITLE,
    "axes.labelsize":        FS_LABEL,
    "xtick.labelsize":       FS_BASE,
    "ytick.labelsize":       FS_BASE,
    "legend.fontsize":       FS_LEGEND,
    "figure.dpi":            DPI_SCREEN,
    "savefig.dpi":           DPI_SAVE,
    "savefig.bbox":          "tight",
    "savefig.pad_inches":    PAD,
    "pdf.fonttype":          42,
    "ps.fonttype":           42,
    "axes.linewidth":        1.2,
    "grid.alpha":            0.4,
    "grid.linewidth":        1.5,
    "grid.linestyle":        "--",
    "axes.titleweight":      "bold",
    "image.cmap":            "viridis",
    "legend.frameon":        True,
    "legend.framealpha":     0.92,
    "legend.edgecolor":      "#CCCCCC",
    "axes.spines.top":       False,
    "axes.spines.right":     False,
    # Tick Mark Styling
    "xtick.major.size":      3,
    "ytick.major.size":      3,
    "xtick.minor.visible":   True,
    "ytick.minor.visible":   True,
    "xtick.minor.size":      1,
    "ytick.minor.size":      1,
})
plt.style.use("seaborn-v0_8-white")


# ─────────────────────────────────────────────────────────────────────────────
# 2. SHARED UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def save_fig(fig, name):
    p = OUT / name
    fig.savefig(p.with_suffix(".pdf"))
    fig.savefig(p.with_suffix(".png"))
    plt.close(fig)
    print(f"  ✓  {name}.pdf / .png")


def panel_label(ax, text, x=0.015, y=0.985, **kw):
    """Bold panel tag in top-left, with a subtle white halo to prevent overlap."""
    d = dict(transform=ax.transAxes, fontsize=FS_PANEL_TAG,
             fontweight="bold", va="top", ha="left", zorder=20)
    d.update(kw)
    t = ax.text(x, y, text, **d)
    t.set_path_effects([
        pe.withStroke(linewidth=3, foreground="white")
    ])
    return t


def stat_box(ax, text, loc="upper right", fs=FS_ANNOT, fc="white",
             ec="#AAAAAA", alpha=0.95, zorder=15):
    """Rounded stat-annotation box, always on top (zorder)."""
    pos = {
        "upper right": (0.97, 0.97, "right", "top"),
        "upper left":  (0.03, 0.97, "left",  "top"),
        "lower right": (0.97, 0.03, "right", "bottom"),
        "lower left":  (0.03, 0.03, "left",  "bottom"),
        "upper centre":(0.50, 0.97, "center","top"),
    }
    x, y, ha, va = pos.get(loc, pos["upper right"])
    ax.text(x, y, text, transform=ax.transAxes,
            fontsize=int(fs), va=va, ha=ha, zorder=zorder,
            bbox=dict(boxstyle="round,pad=0.45", fc=fc,
                      alpha=alpha, ec=ec, lw=0.8))


def lowess_trend(x, y, frac=0.20):
    from statsmodels.nonparametric.smoothers_lowess import lowess
    order = np.argsort(x)
    lw = lowess(y[order], x[order], frac=frac, return_sorted=True)
    return lw[:, 0], lw[:, 1]


def shap_inflection(lx, ly, x_max, ignore_frac=0.05):
    lx_u = np.linspace(float(lx.min()), float(lx.max()), 300)
    f    = interp1d(lx, ly, kind="linear", fill_value="extrapolate")
    ly_u = f(lx_u)
    dy   = np.gradient(ly_u, lx_u)
    n    = len(dy)
    wl   = max(11, min(51, n // 5))
    if wl % 2 == 0: wl += 1
    dy_s = savgol_filter(dy, window_length=wl, polyorder=2)
    skip = int(n * ignore_frac)
    sc   = np.where(np.diff(np.sign(dy_s[skip:])))[0]
    thresh = float(lx_u[skip + sc[0]]) if len(sc) > 0 else float(lx_u.mean())
    return thresh, lx_u, ly_u, dy_s


def categorize(feat):
    f = feat.lower()
    if any(k in f for k in ["temp","lst","nighttime","daytime"]):
        return "Temperature"
    if any(k in f for k in ["cattle","horse","chicken","pig","goat",
                              "sheep","buffalo","duck","livestock"]):
        return "Livestock"
    if any(k in f for k in ["developed","dev_","urban","impervious"]):
        return "Development"
    if any(k in f for k in ["ndvi","evi","savi","vegetation",
                              "forest","grassland","wetland","ndmi"]):
        return "Vegetation"
    if any(k in f for k in ["elevation","dem","topograph"]):
        return "Topography"
    if any(k in f for k in ["water","jrc","permanent","seasonal"]):
        return "Water"
    if any(k in f for k in ["soil"]):
        return "Soil"
    if any(k in f for k in ["corn","crop","soybeans","wheat","cotton",
                              "rice","alfalfa","tobacco","sorghum","fallow"]):
        return "Agriculture"
    if any(k in f for k in ["sar","s1_","vh","vv"]):
        return "SAR"
    return "Other"


def normalize01(arr):
    lo = np.nanpercentile(arr, 1)
    hi = np.nanpercentile(arr, 99)
    out = (arr - lo) / max(hi - lo, 1e-9)
    return np.clip(out, 0, 1)


STATE_ABBREV = {
    "01":"AL","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE",
    "11":"DC","12":"FL","13":"GA","16":"ID","17":"IL","18":"IN","19":"IA",
    "20":"KS","21":"KY","22":"LA","23":"ME","24":"MD","25":"MA","26":"MI",
    "27":"MN","28":"MS","29":"MO","30":"MT","31":"NE","32":"NV","33":"NH",
    "34":"NJ","35":"NM","36":"NY","37":"NC","38":"ND","39":"OH","40":"OK",
    "41":"OR","42":"PA","44":"RI","45":"SC","46":"SD","47":"TN","48":"TX",
    "49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY"
}


# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("Loading results …")
with open(RESULTS_FILE, "rb") as f:
    results = pickle.load(f)

predictions_df  = results["predictions"]
shap_importance = results["shap_importance"]
shap_values_arr = results["shap_values"]
X_sample        = results["shap_sample"]
cv_results_raw  = results["cv_results"]
tables          = results["tables"]
fn_list         = list(X_sample.columns)

if "LE_Bracket" not in predictions_df.columns:
    bins   = [0, 70, 74, 76.5, 79, 200]
    labels = ["60-70","70-74","74-76.5","76.5-79","79-88"]
    predictions_df["LE_Bracket"] = pd.cut(
        predictions_df["actual"], bins=bins, labels=labels)

df_full = pd.read_csv(DATA_FILE)
if "MeanLifeExpectency_x" in df_full.columns:
    df_full["MeanLifeExpectency"] = df_full["MeanLifeExpectency_x"]
    df_full.drop(columns=["MeanLifeExpectency_x","MeanLifeExpectency_y"],
                 inplace=True, errors="ignore")
df_full["fips"] = df_full["fips"].astype(int).astype(str).str.zfill(5)

RENAME = {
    "MeanLifeExpectency":                        "Life Expectancy (Years)",
    "NDVI_mean":                                 "NDVI Mean",
    "NDVI_stdDev":                               "NDVI Variance",
    "LST_Day_1km_mean":                          "Daytime Surface Temp (Mean)",
    "LST_Night_1km_mean":                        "Nighttime Surface Temp (Mean)",
    "LST_Night_1km_p10":                         "Nighttime Temp (10th Percentile)",
    "LST_Day_1km_p90":                           "Daytime Temp (90th Percentile)",
    "LST_Day_1km_stdDev":                        "Daytime Temp Variance",
    "Soil_mean":                                 "Soil Moisture Mean",
    "Soil_stdDev":                               "Soil Moisture Variance",
    "USDA_Cropland_USDA_Forest_Deciduous_pct":   "Deciduous Forest %",
    "USDA_Cropland_USDA_Corn_pct":               "Corn %",
    "mean_cattle":                               "Cattle Density",
    "mean_horse":                                "Horse Density",
    "USDA_Cropland_USDA_Dev_Med_pct":            "Developed (Med Intensity) %",
    "USDA_Cropland_USDA_Dev_Open_pct":           "Developed (Open Space) %",
    "USDA_Cropland_USDA_Dev_Low_pct":            "Developed (Low Intensity) %",
    "USDA_Cropland_USDA_Dev_High_pct":           "Developed (High Intensity) %",
    "USDA_Cropland_USDA_Wetlands_Woody_pct":     "Woody Wetlands %",
    "USDA_Cropland_USDA_Forest_Evergreen_pct":   "Evergreen Forest %",
    "USDA_Cropland_USDA_Wetlands_Herbaceous_pct":"Herbaceous Wetlands %",
    "DEM_p10":                                   "Elevation (10th Percentile)",
}
df_full.rename(columns=RENAME, inplace=True)

county_stats = (predictions_df
                .groupby("fips")
                .agg(MAE      = ("abs_error","mean"),
                     MeanResid= ("residual","mean"),
                     RMSE     = ("residual", lambda x: np.sqrt(np.mean(x**2))))
                .reset_index())

LE_COL = "Life Expectancy (Years)"
if LE_COL not in df_full.columns:
    LE_COL = "MeanLifeExpectency"

print("  Data loaded successfully.\n")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 01a/b/c — SPATIAL MAE MAPS
# fig01a = prediction precision (MAE choropleth)
# fig01b = bias direction (residual choropleth)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 01a/b — SPATIAL MAE MAPS & RESIDUAL BIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig01a/b] Spatial residual maps …")

# DYNAMIC RUN CONSTANTS (Placed here to guarantee they are loaded)
CURRENT_MORANS_I = 0.0988
CURRENT_MORANS_P = 0.0010

try:
    counties = gpd.read_file(COUNTY_SHP)
    states   = gpd.read_file(STATE_SHP)
    counties = counties[counties["STATEFP"].isin(CONTINENTAL)]
    states   = states[states["STATEFP"].isin(CONTINENTAL)]

    geo = counties.merge(county_stats, left_on="GEOID", right_on="fips", how="left")
    geo["cx"] = geo.geometry.centroid.x
    geo["cy"] = geo.geometry.centroid.y
    geo_valid = geo.dropna(subset=["MAE"])

    best5  = geo_valid.nsmallest(5, "MAE")
    worst5 = geo_valid.nlargest(5, "MAE")

    mae_v = county_stats["MAE"].dropna()
    rv    = county_stats["MeanResid"].dropna()
    vmax  = float(np.nanpercentile(mae_v, 95))
    r_lim = min(max(float(np.percentile(np.abs(rv), 95)), 0.05), 4.0)

    # Colorblind-safe marker colors (Wong 2011 palette)
    BEST_COL  = "#009E73"   # teal-green — best counties
    WORST_COL = "#D55E00"   # vermillion — worst counties
    STAR_MS   = 15          # star marker size
    CIRC_MS   = 11          # circle marker size

    def make_county_label(row, col="MAE"):
        state = STATE_ABBREV.get(str(row["GEOID"])[:2], "??")
        val   = row[col]
        return f"{row['NAME']}, {state}  ({val:.2f})"

    # ── FIG 01a: MAE choropleth ────────────────────────────────────────────────
    fig_a, ax = plt.subplots(figsize=(16, 8.5))
    fig_a.suptitle("Spatial Accuracy Portrait: Prediction Precision\n"
                   "Mean Absolute Error per County (CONUS)",
                   fontsize=FS_SUPTITLE, fontweight="bold", y=0.98)

    geo.plot(column="MAE", cmap=CMAP_ERR, ax=ax, vmin=0, vmax=vmax,
             missing_kwds=dict(color=C["lgrey"]), legend=False)
    states.boundary.plot(ax=ax, lw=0.5, edgecolor="white", alpha=0.8)

    sm = cm.ScalarMappable(cmap=CMAP_ERR, norm=mcolors.Normalize(0, vmax))
    sm.set_array([])
    cb = fig_a.colorbar(sm, ax=ax, orientation="horizontal",
                        fraction=0.026, pad=0.03, shrink=0.45)
    cb.set_label("Mean Absolute Error (years)", fontsize=FS_LABEL)
    cb.ax.tick_params(labelsize=FS_BASE)

    # Best: filled star ★ (teal); Worst: filled circle ● (vermillion)
    for _, row in best5.iterrows():
        ax.plot(row["cx"], row["cy"], "*", ms=STAR_MS, color=BEST_COL,
                mec="white", mew=0.8, zorder=10)
    for _, row in worst5.iterrows():
        ax.plot(row["cx"], row["cy"], "o", ms=CIRC_MS, color=WORST_COL,
                mec="white", mew=1.2, zorder=10)

    star_h = mlines.Line2D([0],[0], marker="*", color="w",
                            mfc=BEST_COL, ms=13, label="Top-5 (best predicted)")
    circ_h = mlines.Line2D([0],[0], marker="o", color="w",
                            mfc=WORST_COL, ms=10, label="Worst-5 (highest error)")
    ax.legend(handles=[star_h, circ_h], title="County accuracy",
              fontsize=FS_LEGEND, framealpha=0.95,
              title_fontsize=FS_LEGEND, loc="lower left",
              bbox_to_anchor=(0.01, 0.04))

    best_txt  = "Best-predicted:\n" + "\n".join(
        f"  ★ {make_county_label(r, 'MAE')}" for _, r in best5.iterrows())
    worst_txt = "Hardest:\n" + "\n".join(
        f"  ● {make_county_label(r, 'MAE')}" for _, r in worst5.iterrows())
    ax.text(0.99, 0.04, best_txt + "\n\n" + worst_txt,
            transform=ax.transAxes, fontsize=10.0, va="bottom", ha="right",
            family="monospace", zorder=15,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", alpha=0.93,
                      ec=C["grey"], lw=0.8))

    p50 = (mae_v <= 1.0).mean() * 100
    p87 = (mae_v <= 2.0).mean() * 100
    stat_box(ax,
             f"Median MAE = {mae_v.median():.2f} yrs\n"
             f"≤1 yr: {p50:.0f}% of counties\n"
             f"≤2 yr: {p87:.0f}% of counties",
             loc="upper right", fs=FS_ANNOT)
    ax.axis("off")
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    save_fig(fig_a, "fig01a_spatial_mae_v5")

    # ── FIG 01b: Residual bias choropleth ──────────────────────────────────────
    fig_b2, ax2 = plt.subplots(figsize=(16, 8.5))
    fig_b2.suptitle("Spatial Accuracy Portrait: Bias Direction\n"
                    "Mean Residual per County — Blue=Over-predicted, Red=Under-predicted",
                    fontsize=FS_SUPTITLE, fontweight="bold", y=0.98)

    norm_r = TwoSlopeNorm(vmin=-r_lim, vcenter=0, vmax=r_lim)
    geo.plot(column="MeanResid", cmap=CMAP_DIV, ax=ax2, norm=norm_r,
             missing_kwds=dict(color=C["lgrey"]), legend=False)
    states.boundary.plot(ax=ax2, lw=0.5, edgecolor="white", alpha=0.8)

    sm2 = cm.ScalarMappable(cmap=CMAP_DIV, norm=norm_r)
    sm2.set_array([])
    cb2 = fig_b2.colorbar(sm2, ax=ax2, orientation="horizontal",
                          fraction=0.026, pad=0.03, shrink=0.45)
    cb2.set_label("Mean Residual  (Actual − Predicted, years)", fontsize=FS_LABEL)
    cb2.ax.tick_params(labelsize=FS_BASE)

    over5  = geo_valid.nsmallest(5, "MeanResid")
    under5 = geo_valid.nlargest(5,  "MeanResid")

    # Same best/worst markers on residual map for spatial reference
    for _, row in best5.iterrows():
        ax2.plot(row["cx"], row["cy"], "*", ms=STAR_MS, color=BEST_COL,
                 mec="white", mew=0.8, zorder=10)
    for _, row in worst5.iterrows():
        ax2.plot(row["cx"], row["cy"], "o", ms=CIRC_MS, color=WORST_COL,
                 mec="white", mew=1.2, zorder=10)

    star_h2 = mlines.Line2D([0],[0], marker="*", color="w",
                             mfc=BEST_COL, ms=13, label="Top-5 best MAE")
    circ_h2 = mlines.Line2D([0],[0], marker="o", color="w",
                             mfc=WORST_COL, ms=10, label="Worst-5 highest MAE")
    ax2.legend(handles=[star_h2, circ_h2], title="MAE reference",
               fontsize=FS_LEGEND, framealpha=0.95,
               title_fontsize=FS_LEGEND, loc="lower left",
               bbox_to_anchor=(0.01, 0.04))

    def resid_label(row):
        state = STATE_ABBREV.get(str(row["GEOID"])[:2], "??")
        return f"  {row['NAME']}, {state}  ({row['MeanResid']:+.2f} yr)"

    bias_txt = ("Over-predicted (blue):\n"
                + "\n".join(resid_label(r) for _, r in over5.iterrows())
                + "\n\nUnder-predicted (red):\n"
                + "\n".join(resid_label(r) for _, r in under5.iterrows()))
    ax2.text(0.99, 0.04, bias_txt, transform=ax2.transAxes,
             fontsize=10.0, va="bottom", ha="right", family="monospace", zorder=15,
             bbox=dict(boxstyle="round,pad=0.45", fc="white", alpha=0.93,
                       ec=C["grey"], lw=0.8))
    
    stat_box(ax2,
             f"Mean residual = {rv.mean():.3f} yrs\n"
             f"Skewness = {stats.skew(rv):.2f}\n"
             f"Moran's I ≈ {CURRENT_MORANS_I}  (p = {CURRENT_MORANS_P})",
             loc="upper right", fs=FS_ANNOT)
    ax2.axis("off")
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    save_fig(fig_b2, "fig01b_spatial_residual_v5")

except Exception as e:
    print(f"  ⚠ fig01 error: {e}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 02 — TEMPORAL: merged into 3:1 ratio
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig02] Temporal combined panel …")

t3 = tables["table3_temporal"].reset_index()
years   = t3["year"].values.astype(int)
r2_v    = t3["R2"].values.astype(float)
mae_v   = t3["MAE"].values.astype(float)
mae_std = t3["MAE_Std"].values.astype(float)

rmse_by_yr = (predictions_df.groupby("year")
              .apply(lambda g: np.sqrt(np.mean(g["residual"]**2)))
              .reset_index(name="RMSE"))
rmse_v = rmse_by_yr["RMSE"].values

yr_min, yr_max = int(years.min()), int(years.max())
n_years = yr_max - yr_min + 1

sl_r2,  _, _, p_r2,  _ = stats.linregress(years, r2_v)
sl_mae, _, _, p_mae, _ = stats.linregress(years, mae_v)
sl_rm,  _, _, p_rm,  _ = stats.linregress(years, rmse_v)

# 3:1 ratio using GridSpec
fig = plt.figure(figsize=(16, 6.5))
gs2 = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[3, 1], wspace=0.15)
ax_main = fig.add_subplot(gs2[0, 0])
ax_stat = fig.add_subplot(gs2[0, 1])

fig.suptitle(
    f"Temporal Generalization: Model Metrics Across {n_years} Years ({yr_min}–{yr_max})",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.01)

# Plot MAE and RMSE on primary Y-axis (forced scaling dynamically to fit values)
min_err_ax = min(min(mae_v), min(rmse_v)) * 0.9  # Give some headroom
max_err_ax = max(max(mae_v), max(rmse_v)) * 1.1 
ax_main.set_ylim(min_err_ax, max_err_ax)

# MAE Line
ax_main.plot(years, mae_v, "s-", color=C["red"], lw=2.5, ms=8, mec="white", mew=1.2, zorder=3, label="MAE (years)")
sm_mae = gaussian_filter1d(mae_v, sigma=1)
ax_main.fill_between(years, sm_mae - (mae_std * 0.4), sm_mae + (mae_std * 0.4), alpha=0.15, color=C["red"])
ax_main.plot(years, np.poly1d(np.polyfit(years, mae_v, 1))(years), ":", color=C["red"], lw=1.8, alpha=0.7)

# RMSE Line
ax_main.plot(years, rmse_v, "^-", color=C["purple"], lw=2.5, ms=8, mec="white", mew=1.2, zorder=3, label="RMSE (years)")
ax_main.plot(years, np.poly1d(np.polyfit(years, rmse_v, 1))(years), ":", color=C["purple"], lw=1.8, alpha=0.7)

ax_main.set_xlabel("Year", fontweight="bold")
ax_main.set_ylabel("Error (years) [MAE & RMSE]", fontweight="bold")
ax_main.xaxis.set_major_locator(MultipleLocator(2)) # every 2 years
ax_main.grid(True, alpha=0.22)

# Plot R2 on secondary Y-axis so its ~0.6 values are visible without squashing
ax_r2 = ax_main.twinx()
# Scale R2 to avoid flattening
r2_min_ax = r2_v.min() - 0.1
r2_max_ax = r2_v.max() + 0.1
ax_r2.set_ylim(r2_min_ax, r2_max_ax)
ax_r2.plot(years, r2_v, "o-", color=C["blue"], lw=2.5, ms=8, mec="white", mew=1.2, zorder=4, label="R² (test set)")
sm_r2 = gaussian_filter1d(r2_v, sigma=1)
ax_r2.fill_between(years, sm_r2 - 0.012, sm_r2 + 0.012, alpha=0.15, color=C["blue"])
ax_r2.plot(years, np.poly1d(np.polyfit(years, r2_v, 1))(years), ":", color=C["blue"], lw=1.8, alpha=0.7)
ax_r2.set_ylabel("Explained Variance (R²)", fontweight="bold", color=C["blue"])
ax_r2.tick_params(axis='y', colors=C["blue"])

# Combine legends from both axes
lines_1, labels_1 = ax_main.get_legend_handles_labels()
lines_2, labels_2 = ax_r2.get_legend_handles_labels()
ax_main.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=FS_LEGEND, loc="upper right", framealpha=0.95)

panel_label(ax_main, "Core Metrics Overlay")

# -------------------------------------------------------------------------
# Right panel — BLENDED Legend & Stats 
# -------------------------------------------------------------------------
ax_stat.axis("off")

# Format p-values cleanly for publication
def sig_str(p): 
    if p < 0.001: return "Significant (p < 0.001)"
    elif p < 0.05: return f"Significant (p={p:.3f})"
    else: return f"No trend (p={p:.2f})"

# We use matching Unicode symbols (●, ■, ▲) to replace the plot legend entirely!
stat_entries = [
    ("■ MAE (years)", "Mean Absolute Error", C["red"], sl_mae, p_mae),
    ("▲ RMSE (years)", "Root Mean Squared Error", C["purple"], sl_rm, p_rm),
    ("● R² (test set)", "Explained Variance", C["blue"], sl_r2, p_r2),
]
y_pos_stat = [0.85, 0.50, 0.15]

for (marker_text, metric_name, col, sl, pv), yp in zip(stat_entries, y_pos_stat):
    text_content = f"{marker_text}\n{metric_name}\nTrend: {sl:+.4f}/yr\n{sig_str(pv)}"
    
    # We use a slightly thicker colored edge (lw=2.5) to act as the color key
    ax_stat.text(0.5, yp, text_content,
                 transform=ax_stat.transAxes,
                 fontsize=FS_ANNOT+1, va="center", ha="center",
                 color="#222", fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.8", fc="white", alpha=0.95, ec=col, lw=2.5))

plt.tight_layout()
save_fig(fig, "fig02_temporal_combined_v5")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 03 — SHAP BEESWARM
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig03] SHAP beeswarm …")

N_TOP = 20
top_idx  = shap_importance.head(N_TOP).index.tolist()
top_feat = shap_importance.loc[top_idx, "Feature"].tolist()

col_idx, valid_feat = [], []
for fn in top_feat:
    if fn in fn_list:
        col_idx.append(fn_list.index(fn))
        valid_feat.append(fn)

shap_top = shap_values_arr[:, col_idx]
X_top    = X_sample[valid_feat]
cats     = [categorize(f) for f in valid_feat]
cat_cols = [CAT_PAL[c] for c in cats]

fig, ax = plt.subplots(figsize=(14, 10.5))
shap.summary_plot(shap_top, X_top, feature_names=valid_feat,
                  max_display=N_TOP, show=False, plot_type="dot",
                  color_bar=True,
                  color_bar_label="Feature value (normalized)",
                  alpha=0.55, plot_size=None)

for tick, col in zip(ax.get_yticklabels(), reversed(cat_cols)):
    tick.set_color(col)
    tick.set_fontsize(FS_LABEL)
    tick.set_fontweight("semibold")

ax.set_xlabel("SHAP value  (impact on predicted life expectancy, years)",
              fontweight="bold", fontsize=FS_LABEL)
ax.set_title("SHAP Feature Importance: Top-20 Predictors\n"
             "Tick colour = feature category",
             fontsize=FS_SUPTITLE, fontweight="bold", loc="left", pad=12)
ax.axvline(0, color="#444", lw=0.8, ls="--", alpha=0.5)
ax.grid(axis="x", alpha=0.2)

present_cats = sorted(set(cats), key=lambda k: list(CAT_PAL.keys()).index(k))
leg_patches = [mpatches.Patch(color=CAT_PAL[k], label=k) for k in present_cats]
ax.legend(handles=leg_patches, title="Feature category",
          loc="lower right", framealpha=0.95,
          fontsize=FS_LEGEND, title_fontsize=FS_LEGEND, ncol=1)
plt.tight_layout()
save_fig(fig, "fig03_shap_beeswarm_v5")




# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 04a and 04b — PHYSICS & BREAKPOINTS  (MERGED: metabolic + raw-vs-SHAP)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 04a — METABOLIC BREAKPOINTS (INFLECTION + THRESHOLD)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig04a] Metabolic breakpoints — Inflection vs Threshold …")

BREAK_FEATS = [
    ("Pig Density",              "Pig density (head/km²)",     "#984ea3", "RdPu",   "(A)", "(D)"),
    ("Developed (Open Space) %", "Developed (Open Space) (%)", "#377eb8", "Blues",  "(B)", "(E)"),
    ("Cattle Density",           "Cattle density (head/km²)",  "#ff7f00", "YlOrRd", "(C)", "(F)"),
]

fig = plt.figure(figsize=(24, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.52, wspace=0.34)

ax_sc = [fig.add_subplot(gs[0, i]) for i in range(3)]
ax_dv = [fig.add_subplot(gs[1, i]) for i in range(3)]

fig.suptitle(
    "Non-Linear Metabolic Breakpoints: The Diminishing Returns Framework\n"
    "Inflection = where derivative crosses zero | Threshold = saturation elbow (diminishing returns)",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.01)

# ====================== CORRECT DUAL FINDER ======================
def find_inflection_and_threshold(lx, ly, dy_s):
    skip = int(len(lx) * 0.02)
    
    # 1. Inflection = first point where derivative crosses zero (from + to - or early - to + for cattle)
    inflection = np.nan
    crossings = np.where(np.diff(np.sign(dy_s[skip:])))[0]
    for c in crossings:
        idx = skip + c
        # Prefer + → - (true harm), but accept early - → + for cattle
        if (dy_s[idx] > 0 and dy_s[idx + 1] <= 0) or (dy_s[idx] < 0 and dy_s[idx + 1] > 0):
            inflection = float(lx[idx])
            break
    
    # 2. Threshold = Elbow (Kneedle) - point of strongest diminishing returns
    xn = (lx[skip:] - lx[skip:].min()) / (lx[skip:].max() - lx[skip:].min() + 1e-9)
    yn = (ly[skip:] - ly[skip:].min()) / (ly[skip:].max() - ly[skip:].min() + 1e-9)
    p1 = np.array([xn[0], yn[0]])
    p2 = np.array([xn[-1], yn[-1]])
    points = np.column_stack((xn, yn))
    dists = np.abs(np.cross(p2 - p1, p1 - points)) / np.linalg.norm(p2 - p1)
    elbow_idx = np.argmax(dists)
    threshold = float(lx[skip + elbow_idx])

    # Force logical order: Inflection usually earlier than Threshold
    if not np.isnan(inflection) and inflection > threshold:
        inflection, threshold = threshold, inflection

    return inflection, threshold

TRIM_PCT = 0.0

for i, (feat, xlabel, col, cmap, tag_s, tag_d) in enumerate(BREAK_FEATS):
    if feat not in fn_list:
        ax_sc[i].text(0.5, 0.5, f"'{feat}' not in SHAP sample", ha="center", va="center", transform=ax_sc[i].transAxes)
        continue

    fidx = fn_list.index(feat)
    xv = np.asarray(X_sample[feat].values)
    yv = np.asarray(shap_values_arr[:, fidx])
    
    nz_thresh = np.nanpercentile(xv[xv>0], TRIM_PCT) if (xv>0).sum() > 10 else 0
    nz = xv > nz_thresh
    xfit = xv[nz] if nz.sum() > 100 else xv
    yfit = yv[nz] if nz.sum() > 100 else yv
    frac = 0.30 if i == 0 else 0.40

    sc_ = ax_sc[i].scatter(xv, yv, c=xv, cmap=cmap, alpha=0.28, s=12, linewidths=0, rasterized=True)
    plt.colorbar(sc_, ax=ax_sc[i], label=xlabel, fraction=0.03, pad=0.02)

    lx, ly = lowess_trend(xfit, yfit, frac=frac)
    ax_sc[i].plot(lx, ly, color=col, lw=3, zorder=6, label="LOWESS trend")

    lx_u = np.linspace(float(lx.min()), float(lx.max()), 300)
    f = interp1d(lx, ly, kind="linear", fill_value="extrapolate")
    ly_u = f(lx_u)
    dy = np.gradient(ly_u, lx_u)
    
    wl = max(11, min(51, len(dy) // 5))
    if wl % 2 == 0: wl += 1
    dy_s = savgol_filter(dy, window_length=wl, polyorder=2)

    inflection, threshold = find_inflection_and_threshold(lx_u, ly_u, dy_s)
    has_inflect = not np.isnan(inflection)

    is_pct = float(np.nanmax(xv)) <= 1.5
    fmt_inf = f"{int(inflection)}" if has_inflect else "None"
    fmt_thr = f"{int(threshold)}" if not is_pct else f"{threshold:.2f}"

    xmn, xmx = float(xv.min()), float(xv.max())
    ymn, ymx = float(yv.min()) - 0.05, float(yv.max()) + 0.05
    h = ymx - ymn

    # Shading: Beneficial up to Inflection, then Diminishing, then Harmful if inflection exists
    ax_sc[i].add_patch(Rectangle((xmn, ymn), (inflection if has_inflect else threshold) - xmn, h,
                                  color=C["green"], alpha=0.07, label="Beneficial"))
    
    if has_inflect:
        ax_sc[i].add_patch(Rectangle((inflection, ymn), threshold - inflection if threshold > inflection else xmx - inflection, h,
                                      color=C["orange"], alpha=0.07, label="Diminishing"))
        if threshold > inflection:
            ax_sc[i].add_patch(Rectangle((threshold, ymn), xmx - threshold, h,
                                          color=C["red"], alpha=0.08, label="Harmful"))
    else:
        ax_sc[i].add_patch(Rectangle((threshold, ymn), xmx - threshold, h,
                                      color=C["orange"], alpha=0.07, label="Diminishing"))

    # Vertical lines
    if has_inflect:
        ax_sc[i].axvline(inflection, color="#2ca02c", ls="--", lw=2.5, zorder=8, label=f"Inflection = {fmt_inf}")
    ax_sc[i].axvline(threshold, color="#d62728", ls="-", lw=2.8, zorder=7, label=f"Threshold = {fmt_thr}")

    ax_sc[i].axhline(0, color="#999", lw=0.7)
    ax_sc[i].set_xlim(xmn, xmx)
    ax_sc[i].set_ylim(ymn, ymx)
    ax_sc[i].set_xlabel(xlabel, fontweight="bold")
    ax_sc[i].set_ylabel("SHAP value  (years)", fontweight="bold")
    panel_label(ax_sc[i], f"{tag_s}  {feat}", y=1.12)

    # Legend
    leg_b = mpatches.Patch(color=C["green"], alpha=0.3, label="Beneficial")
    leg_d = mpatches.Patch(color=C["orange"], alpha=0.3, label="Diminishing")
    leg_h = mpatches.Patch(color=C["red"], alpha=0.4, label="Harmful")
    line_inf = mlines.Line2D([0], [0], color="#2ca02c", ls="--", lw=2.5, label=f"Inflection = {fmt_inf}")
    line_thr = mlines.Line2D([0], [0], color="#d62728", ls="-", lw=2.8, label=f"Threshold = {fmt_thr}")
    
    handles = [leg_b, leg_d]
    if has_inflect:
        handles += [leg_h, line_inf]
    handles.append(line_thr)
    
    ax_sc[i].legend(handles=handles, fontsize=FS_LEGEND-1.5, framealpha=0.95, loc="lower right")
    ax_sc[i].grid(True, alpha=0.2)

    # ====================== DERIVATIVE PANEL ======================
    ax_dv[i].bar(lx_u, dy_s, width=(lx_u[1]-lx_u[0])*0.9,
                 color=[C["green"] if v >= 0 else C["red"] for v in dy_s], alpha=0.55)
    ax_dv[i].plot(lx_u, dy_s, color=col, lw=1.8, zorder=5)
    ax_dv[i].axhline(0, color="#444", lw=1.2)

    if has_inflect:
        ax_dv[i].axvline(inflection, color="#2ca02c", ls="--", lw=2.5)
    ax_dv[i].axvline(threshold, color="#d62728", ls="-", lw=2.8)

    # Annotation box
    box_txt = f"Inflection: {fmt_inf}" if has_inflect else ""
    box_txt += f"\nThreshold: {fmt_thr}" if box_txt else f"Threshold: {fmt_thr}"
    ax_dv[i].text(0.96, 0.88, box_txt.strip(), transform=ax_dv[i].transAxes,
                  fontsize=FS_ANNOT, va="top", ha="right", fontweight="bold",
                  bbox=dict(fc="white", alpha=0.92, ec="#555", lw=1))

    gp = mpatches.Patch(color=C["green"], alpha=0.6, label="Positive rate")
    rp = mpatches.Patch(color=C["red"], alpha=0.6, label="Negative rate")
    ax_dv[i].legend(handles=[gp, rp], fontsize=FS_LEGEND-1, framealpha=0.95, loc="lower right")
    
    ax_dv[i].set_xlabel(xlabel, fontweight="bold")
    ax_dv[i].set_ylabel("d(SHAP)/d(feature)", fontweight="bold")
    panel_label(ax_dv[i], f"{tag_d}  Derivative Rates", y=1.12)
    ax_dv[i].grid(True, alpha=0.2)
    ax_dv[i].set_xlim(xmn, xmx)

plt.tight_layout()
save_fig(fig, "fig04a_metabolic_breakpoints_v5")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 04b — PHYSICS COUPLING: RAW ENVIRONMENTAL SIGNAL vs SHAP-ATTRIBUTED EFFECT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

print("[fig04b] Physics coupling …")
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
from scipy.stats import pearsonr
from matplotlib.transforms import blended_transform_factory

# --- 1. DYNAMIC ROBUST MATCHING FUNCTION ---
def get_real_feat_name(hints, columns):
    """Scans columns and returns the first match that contains any hint (case-insensitive)."""
    for hint in hints:
        for col in columns:
            if hint.lower().replace(" ", "") in col.lower().replace(" ", ""):
                return col
    return hints[0] # Fallback if nothing matches

# --- 2. STRICT KEYWORD HINTS BASED ON SHAP RANKINGS ---
# We use exact names to prevent the raw data and SHAP data from grabbing different features
PHYS_PAIRS_HINTS = [
    # Row 1: Nighttime (Rank 2)
    (["Nighttime Temp (10th Percentile)"], 
     "Nighttime LST (10th Pct, °C)", 
     C["red"]),
    
    # Row 2: Vegetation (Rank 8) - Pure NDVI was pruned, so TVI is the strongest proxy
    (["Thermal Vegetation Index (TVI)"], 
     "Thermal Vegetation Index (TVI)", 
     C["green"]),
    
    # Row 3: Daytime (Rank 15) - Apples-to-apples comparison with Nighttime 10th Pct
    (["Daytime Temp (10th Percentile)"], 
     "Daytime LST (10th Pct, °C)", 
     C["orange"]),
]

fig4b = plt.figure(figsize=(18, 15)) 
fig4b.suptitle(
    "Physics of Life Expectancy: Raw Environmental Signal vs. Model Attribution\n"
    "Left: raw data density — Right: SHAP-attributed effect (years)",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.04) 
gs4b = gridspec.GridSpec(3, 2, figure=fig4b, hspace=0.52, wspace=0.34)
letters = ["(A)","(B)","(C)","(D)","(E)","(F)"]
pi = 0

for row_i, (hints, disp_name, col) in enumerate(PHYS_PAIRS_HINTS):
    ax_r  = fig4b.add_subplot(gs4b[row_i, 0])
    ax_sh = fig4b.add_subplot(gs4b[row_i, 1])
    
    # --- 3. DYNAMICALLY RESOLVE EXACT FEATURE NAMES ---
    raw_feat = get_real_feat_name(hints, df_full.columns)
    shap_feat = get_real_feat_name(hints, fn_list)
    
    # Check if we are dealing with a vegetation index to adjust scaling
    is_veg = any(v in raw_feat.lower() for v in ["ndvi", "tvi", "vegetation", "evi", "savi"])

    # ==========================================
    # LEFT PANEL: RAW HEXBIN
    # ==========================================
    if raw_feat in df_full.columns and LE_COL in df_full.columns:
        tmp = df_full[[raw_feat, LE_COL]].dropna()
        xr  = np.asarray(tmp[raw_feat].values)
        
        # Scale vegetation indices if they are raw integer scaled
        if is_veg:
            veg_sc = 10000.0 if xr.max() > 10 else 1.0
            xr = xr / veg_sc
            
        xn  = normalize01(xr)
        yr_ = tmp[LE_COL].values
        
        hb  = ax_r.hexbin(xn, yr_, gridsize=42, cmap="viridis",
                           mincnt=1, alpha=0.92, linewidths=0.15)
        cb  = fig4b.colorbar(hb, ax=ax_r, shrink=0.75, pad=0.02)
        cb.set_label("County-year density", fontsize=FS_ANNOT)
        cb.ax.tick_params(labelsize=FS_ANNOT)
        
        try:
            ord_ = np.argsort(xn)
            lw_  = sm_lowess(yr_[ord_], xn[ord_], frac=0.25, return_sorted=True)
            ax_r.plot(lw_[:,0], lw_[:,1], color="white", lw=2.8, ls="--", zorder=5)
            ax_r.plot(lw_[:,0], lw_[:,1], color=col,    lw=1.8, ls="--",
                      zorder=6, alpha=0.85, label="LOWESS trend")
        except Exception:
            pass
            
        rv2, pv2 = pearsonr(xn, yr_)
        xlab = ("Index Value (0=low, 1=high)" if is_veg else f"{disp_name} (normalized 0–1)")
        
        ax_r.set_xlabel(xlab, fontweight="bold", fontsize=FS_LABEL)
        ax_r.set_ylabel("Life expectancy  (years)", fontweight="bold", fontsize=FS_LABEL)
        ax_r.set_xlim(-0.02, 1.02)
        ax_r.grid(True, alpha=0.18)
        
        clean_raw_title = raw_feat.replace("_", " ")
        ax_r.set_title(f"{letters[pi]}  {clean_raw_title}", fontweight="bold", loc="left",
                       fontsize=FS_TITLE, pad=12)
                       
        stat_box(ax_r,
                 f"Pearson r = {rv2:.3f}  (p<0.001)\n"
                 f"n = {len(tmp):,} county-years",
                 loc="lower right", fs=FS_ANNOT,
                 fc="#FFFDE7" if rv2 < 0 else "white")
                 
        ax_r.legend(fontsize=FS_LEGEND-1, loc="upper right", framealpha=0.92)
    else:
        print(f"⚠️ WARNING: Could not find raw feature matching {hints[0]} in df_full")
        ax_r.set_title(f"{letters[pi]} MISSING: {hints[0]}")
        
    pi += 1

    # ==========================================
    # RIGHT PANEL: SHAP ATTRIBUTION
    # ==========================================
    if shap_feat in fn_list:
        sidx  = fn_list.index(shap_feat)
        xs    = np.asarray(X_sample[shap_feat].values.copy())
        ys    = np.asarray(shap_values_arr[:, sidx])
        
        if is_veg and xs.max() > 10:
            xs = xs / 10000.0
            
        sc_ = ax_sh.scatter(xs, ys, c=xs, cmap="RdYlGn" if is_veg else "coolwarm",
                             alpha=0.45, s=14, linewidths=0, rasterized=True)
        cb2 = fig4b.colorbar(sc_, ax=ax_sh, shrink=0.75, pad=0.02)
        cb2.set_label("Feature value", fontsize=FS_ANNOT)
        cb2.ax.tick_params(labelsize=FS_ANNOT)
        
        try:
            lx2, ly2 = lowess_trend(xs, ys, frac=0.20)
            ax_sh.plot(lx2, ly2, color=col, lw=2.8, zorder=5, label="LOWESS")
            sc_idx = np.where(np.diff(np.sign(ly2)))[0]
            
            thresholds = [float(lx2[sci]) for sci in sc_idx[:2]] 
            unit = "" if is_veg else " °C"
            
            if len(thresholds) > 0:
                for xc in thresholds:
                    ax_sh.axvline(xc, color="#333", lw=1.8, ls=":", alpha=0.8)
                
                thr_str = " & ".join([f"{x:.2f}{unit}" for x in thresholds])
                lbl = f"Threshold\n@ {thr_str}" if len(thresholds) == 1 else f"Thresholds\n@ {thr_str}"
                
                if len(thresholds) == 1:
                    ann_x = thresholds[0] + (xs.max() - xs.min()) * 0.04
                    ha_ = "left"
                else:
                    ann_x = np.mean(thresholds)
                    ha_ = "center"
                
                # Dynamic Y-placement to prevent overlap with the LOWESS curve
                curve_mid_y = np.median(ly2)
                ann_y_axis_coords = 0.75 if curve_mid_y < 0 else 0.25 
                
                trans_popout = blended_transform_factory(ax_sh.transData, ax_sh.transAxes)
                
                ax_sh.text(ann_x, ann_y_axis_coords, lbl, transform=trans_popout,
                           fontsize=FS_ANNOT, color="#333", ha=ha_, va="center",
                           bbox=dict(boxstyle="round,pad=0.45", fc="white", alpha=0.85, ec=C["grey"], lw=1.5, zorder=12))
        except Exception:
            pass
            
        ax_sh.axhline(0, color="#666", lw=1.0, ls="--", alpha=0.6, label="Zero SHAP")
        xlab2 = ("Index Value (0=bare, 1=dense veg)" if is_veg else f"{disp_name} (raw °C)")
        
        ax_sh.set_xlabel(xlab2, fontweight="bold", fontsize=FS_LABEL)
        ax_sh.set_ylabel("SHAP value  (Δ LE, years)", fontweight="bold", fontsize=FS_LABEL)
        ax_sh.grid(True, alpha=0.18)
        
        clean_shap_title = shap_feat.replace("_", " ")
        ax_sh.set_title(f"{letters[pi]}  SHAP attribution: {clean_shap_title}", fontweight="bold",
                        loc="left", fontsize=FS_TITLE, pad=12)
                        
        stat_box(ax_sh,
                 f"Mean |SHAP| = {float(np.mean(np.abs(ys))):.3f} yr",
                 loc="upper right", fs=FS_ANNOT)
        ax_sh.legend(fontsize=FS_LEGEND-1, framealpha=0.92, loc="lower left")
    else:
        print(f"⚠️ WARNING: Could not find SHAP feature matching {hints[0]} in fn_list")
        ax_sh.set_title(f"{letters[pi]} MISSING SHAP: {hints[0]}")
        
    pi += 1

fig4b.text(0.5, 0.02,
           "Physics interpretation: nighttime LST = chronic radiative heat load; "
           "TVI = vegetation-albedo-cooling feedback; "
           "daytime LST = peak energy-balance stress.\n"
           "SHAP values confirm causal dominance after controlling for other features.",
           ha="center", fontsize=FS_ANNOT, style="italic", color="#444")

save_fig(fig4b, "fig04b_physics_coupling_v5")



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 05 — FOREST BUFFER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig05] Forest buffer …")

LST_FEAT    = "Daytime Surface Temp (Mean)"
FOREST_FEAT = "Deciduous Forest %"
PAL_F = {"Low (<5%)":C["red"], "Medium (5–20%)":C["orange"], "High (>20%)":C["green"]}
LABS_F = ["Low (<5%)", "Medium (5–20%)", "High (>20%)"]
BINS_F = [-np.inf, 0.05, 0.20, np.inf]

fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))

if LST_FEAT in fn_list and FOREST_FEAT in fn_list:
    li  = fn_list.index(LST_FEAT)
    lst_x = np.asarray(X_sample[LST_FEAT].values)
    lst_s = np.asarray(shap_values_arr[:, li])
    fv    = np.asarray(X_sample[FOREST_FEAT].values)
    fcat  = pd.cut(fv, bins=BINS_F, labels=LABS_F)

    ax = axes[0]
    for cat in LABS_F:
        mask = fcat == cat
        ax.scatter(lst_x[mask], lst_s[mask], c=PAL_F[cat],
                   alpha=0.25, s=15, label=cat, rasterized=True, linewidths=0)
        if mask.sum() > 80:
            lxc, lyc = lowess_trend(lst_x[mask], lst_s[mask])
            ax.plot(lxc, lyc, color=PAL_F[cat], lw=2.8, zorder=5)

    ax.axhline(0, color="#999", lw=2.0)
    ax.set_xlabel("Daytime LST (°C)", fontweight="bold")
    ax.set_ylabel("LST SHAP value  (years)", fontweight="bold")
    
    ax.set_title("(A)  Stratum-stratified SHAP effect of daytime LST\n"
                 "Lines diverge at high temperatures → buffering",
                 fontweight="bold", loc="left", fontsize=FS_TITLE, pad=12)
                 
    leg_a = [mlines.Line2D([0], [0], color=PAL_F[c], lw=3, label=c) for c in LABS_F]
    ax.legend(handles=leg_a, title="Deciduous forest cover",
              fontsize=FS_LEGEND, framealpha=0.95, title_fontsize=FS_LEGEND,
              loc="lower right")
              
    ax.grid(True, alpha=0.2)

    ax2 = axes[1]
    q_labels = ["Q1\n(cool)", "Q2", "Q3", "Q4\n(hot)"]
    tq   = pd.qcut(lst_x, q=4, labels=q_labels)
    df_i = pd.DataFrame({"tq":tq, "fc":fcat, "sv":lst_s}).dropna()
    pos  = np.array([0,1,2,3])
    offs = {"Low (<5%)":-0.28, "Medium (5–20%)":0, "High (>20%)":0.28}
    W    = 0.25

    for cat in LABS_F:
        sub = df_i[df_i["fc"]==cat]
        bd  = [sub[sub["tq"]==q]["sv"].values for q in q_labels]
        ax2.boxplot(bd, positions=pos+offs[cat], widths=W,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color="white", lw=1.8),
                    boxprops=dict(facecolor=PAL_F[cat], alpha=0.78),
                    whiskerprops=dict(color=PAL_F[cat]),
                    capprops=dict(color=PAL_F[cat]))

    q4l = df_i[(df_i["fc"]=="Low (<5%)") & (df_i["tq"]=="Q4\n(hot)")]["sv"].median()
    q4h = df_i[(df_i["fc"]=="High (>20%)") & (df_i["tq"]=="Q4\n(hot)")]["sv"].median()
    gap = q4l - q4h
    x4  = pos[3]
    
    # --- DYNAMIC CALCULATION ---
    atten_pct = abs(gap / q4l) * 100 if q4l != 0 else 0

    # --- SET TITLES DYNAMICALLY ---
    fig.suptitle("Forest as a Life-Expectancy Buffer Against Heat Stress\n"
                 f"High-forest counties suffer ~{atten_pct:.0f}% less LST penalty in the hottest quartile",
                 fontsize=FS_SUPTITLE, fontweight="bold", y=1.06)

    stat_box(ax, f"High forest (>20%)\nattenuates heat penalty\nby ~{atten_pct:.0f}% in hottest counties",
             loc="upper right", fs=FS_ANNOT, fc="#E8F5E9")

    ax2.annotate("", xy=(x4+0.28, q4h), xytext=(x4-0.28, q4l),
                 arrowprops=dict(arrowstyle="<->", color="#333", lw=2.5))
    
    ax2.text(x4+0.38, (q4l+q4h)/2,
             f"Δ={gap:.3f} yr\n(buffering gap)",
             fontsize=FS_ANNOT, va="center", color="#333",
             bbox=dict(fc="white", alpha=0.9, ec=C["grey"], lw=1.0))

    ax2.set_xticks(pos)
    ax2.set_xticklabels(q_labels)
    ax2.set_xlabel("Temperature quartile", fontweight="bold")
    ax2.set_ylabel("LST SHAP value  (years)", fontweight="bold")
    
    ax2.set_title("(B)  Quantified buffering effect by temperature quartile\n"
                  "↔ gap in Q4 = forest-driven LE protection",
                  fontweight="bold", loc="left", fontsize=FS_TITLE, pad=12)
                  
    ax2.axhline(0, color="#999", lw=2.0)
    ax2.grid(axis="y", alpha=0.2)
    
    leg_p = [mpatches.Patch(color=PAL_F[c], label=c, alpha=0.85) for c in LABS_F]
    ax2.legend(handles=leg_p, title="Forest cover",
               fontsize=FS_LEGEND, framealpha=0.95, title_fontsize=FS_LEGEND,
               loc="upper center", ncol=3)

plt.tight_layout()
save_fig(fig, "fig05_forest_buffer_v5")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 06 — SOIL GRADIENT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig06] Soil gradient …")

SOIL_COL = "Soil Moisture Mean"
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
fig.suptitle("Soil Health Gradient: 'Wealth from Dirt'\n"
             "Optimal soil moisture (6–8) correlates with longer life expectancy",
             fontsize=FS_SUPTITLE, fontweight="bold", y=1.03)

if SOIL_COL in df_full.columns and LE_COL in df_full.columns:
    df_s = df_full[[SOIL_COL, LE_COL]].dropna()
    df_s = df_s[(df_s[SOIL_COL] > df_s[SOIL_COL].quantile(0.01)) &
                (df_s[SOIL_COL] < df_s[SOIL_COL].quantile(0.99))]

    ax = axes[0]
    hb = ax.hexbin(df_s[SOIL_COL], df_s[LE_COL],
                   gridsize=45, cmap="Blues", mincnt=1, linewidths=0.2)
    cb_ = plt.colorbar(hb, ax=ax, fraction=0.03, pad=0.02)
    cb_.set_label("County-year count", fontsize=FS_ANNOT)
    cb_.ax.tick_params(labelsize=FS_ANNOT)
    lx_s, ly_s = lowess_trend(df_s[SOIL_COL].values, df_s[LE_COL].values)
    ax.plot(lx_s, ly_s, color=C["orange"], lw=2.5, label="LOWESS trend")

    q_b = pd.qcut(df_s[SOIL_COL], q=4,
                  labels=["Q1\n(driest)","Q2","Q3","Q4\n(wettest)"])
    qst = df_s.groupby(q_b, observed=True)[LE_COL].agg(["median","std"])
    qmd = df_s.groupby(q_b, observed=True)[SOIL_COL].median()
    ax.errorbar(qmd, qst["median"], yerr=qst["std"], fmt="D",
                color=C["orange"], ms=9, capsize=4, lw=1.8,
                mec="white", mew=1.2, zorder=7, label="Quartile median ± SD")

    ylo = float(df_s[LE_COL].min())-1
    yhi = float(df_s[LE_COL].max())+1
    ax.add_patch(Rectangle((6, ylo), 2, yhi-ylo,
                            color=C["green"], alpha=0.10, zorder=0,
                            label="Optimal zone (SM=6–8)"))
    r_, pv_ = stats.pearsonr(df_s[SOIL_COL], df_s[LE_COL])
    ax.set_xlabel("Soil moisture mean  (ESA CCI ×100;\n~6–8 ≈ field capacity)",
                  fontweight="bold", fontsize=FS_LABEL)
    ax.set_ylabel("Life expectancy  (years)", fontweight="bold")
    panel_label(ax, "(A)", y=0.985)
    ax.set_title("Raw relationship: optimal band 6–8 supports longest LE",
                 fontweight="bold", loc="left", fontsize=FS_TITLE)
    ax.legend(fontsize=FS_LEGEND, framealpha=0.95, loc="upper right")
    ax.grid(True, alpha=0.2)
    stat_box(ax, f"r = {r_:.3f}  (p<0.001)\nn = {len(df_s):,} county-years",
             loc="lower right", fs=FS_ANNOT)

if SOIL_COL in fn_list:
    ax2 = axes[1]
    si  = fn_list.index(SOIL_COL)
    sv_ = np.asarray(X_sample[SOIL_COL].values)
    ss_ = np.asarray(shap_values_arr[:, si])
    sc2 = ax2.scatter(sv_, ss_, c=sv_, cmap="Blues_r", alpha=0.38, s=13,
                      linewidths=0, rasterized=True)
    cb2 = plt.colorbar(sc2, ax=ax2, fraction=0.03, pad=0.02)
    cb2.set_label("Soil moisture value", fontsize=FS_ANNOT)
    cb2.ax.tick_params(labelsize=FS_ANNOT)
    lx2, ly2 = lowess_trend(sv_, ss_)
    ax2.plot(lx2, ly2, color=C["orange"], lw=2.5, label="LOWESS trend")
    ax2.add_patch(Rectangle((6, float(ss_.min())-0.05), 2,
                             float(ss_.max())-float(ss_.min())+0.1,
                             color=C["green"], alpha=0.10, zorder=0,
                             label="Optimal zone (SM=6–8)"))
    ax2.axhline(0, color="#999", lw=2.0)
    pk = int(np.argmax(ly2))
    ax2.plot(float(lx2[pk]), float(ly2[pk]), "o", ms=9, color=C["orange"],
             mec="white", mew=1.2, zorder=7)
    # Peak annotation: place it inside optimal zone with clear offset
    ax2.annotate(f"Peak SHAP\n@ SM={float(lx2[pk]):.1f}",
                 xy=(float(lx2[pk]), float(ly2[pk])),
                 xytext=(float(lx2[pk])-1.2, float(ly2[pk])+0.08),
                 fontsize=FS_ANNOT, color="#333",
                 bbox=dict(fc="white", alpha=0.92, ec=C["grey"], lw=0.6),
                 arrowprops=dict(arrowstyle="->", color="#888", lw=0.9))
    # Waterlogging label: bottom right corner
    ax2.text(0.97, 0.05,
             "SM>8.5: waterlogging\n→ model penalises",
             transform=ax2.transAxes,
             fontsize=FS_ANNOT, color=C["red"], ha="right", va="bottom",
             bbox=dict(fc="#FFF0EE", alpha=0.92, ec=C["red"], lw=2.0))
    ax2.set_xlabel("Soil moisture mean  (ESA CCI ×100)", fontweight="bold",
                   fontsize=FS_LABEL)
    ax2.set_ylabel("SHAP value  (Δ LE, years)", fontweight="bold")
    panel_label(ax2, "(B)", y=0.985)
    ax2.set_title("Model-attributed soil effect: inverted-U shape",
                  fontweight="bold", loc="left", fontsize=FS_TITLE)
    ax2.legend(fontsize=FS_LEGEND, framealpha=0.95, loc="upper right")
    ax2.grid(True, alpha=0.2)

plt.tight_layout()
save_fig(fig, "fig06_soil_gradient_v5")



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 07 — ABLATION / MASTER SUMMARY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig07] Ablation / master summary …")

# Dynamically load the ablation table directly from the CSV
ABLATION_CSV = Path('/Users/faizahmad/Desktop/paper1/paper1_run1/results_individual_modalities_final/master_summary_formatted.csv')

if ABLATION_CSV.exists():
    ab_raw = pd.read_csv(ABLATION_CSV)
    # Parse the "0.6037 ± 0.0433" string formats into numeric columns
    ab = pd.DataFrame({
        "Modality": ab_raw["Modality"],
        "N_Features": ab_raw["N_Features"],
        "R2_mean": ab_raw["R2"].astype(str).str.split(" ± ").str[0].astype(float),
        "R2_std": ab_raw["R2"].astype(str).str.split(" ± ").str[1].astype(float),
        "MAE_mean": ab_raw["MAE (years)"].astype(str).str.split(" ± ").str[0].astype(float),
        "MAE_std": ab_raw["MAE (years)"].astype(str).str.split(" ± ").str[1].astype(float)
    })
else:
    raise FileNotFoundError(f"Ablation CSV not found at {ABLATION_CSV}! Cannot plot Fig 07.")

ab["label"] = [f"{m}\n(n={n})" for m, n in zip(ab["Modality"], ab["N_Features"])]
n  = len(ab)
x  = np.arange(n)

r2_norm  = mcolors.Normalize(ab["R2_mean"].min(),  ab["R2_mean"].max())
mae_norm = mcolors.Normalize(ab["MAE_mean"].min(), ab["MAE_mean"].max())
cmap_v   = cm.get_cmap("viridis")
r2_cols  = [cmap_v(r2_norm(v)) for v in ab["R2_mean"]]
mae_cols = [cmap_v(1.0 - mae_norm(v)) for v in ab["MAE_mean"]]

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
fig.suptitle("Model Modalities: Master Summary (R² ± SD & MAE ± SD)\n"
             "Multimodal fusion markedly outperforms single-modality alternatives",
             fontsize=FS_SUPTITLE, fontweight="bold", y=1.03)

# Panel A — R² 
ax = axes[0]
bars_ = ax.bar(x, ab["R2_mean"], yerr=ab["R2_std"], capsize=6,
               color=r2_cols, edgecolor="white", lw=0.9, alpha=0.98, zorder=3)
ylim_top = (ab["R2_mean"]+ab["R2_std"]).max() * 1.15
ax.set_ylim(0, ylim_top)
ax.set_xticks(x)
ax.set_xticklabels(ab["label"], rotation=40, ha="right", fontsize=FS_BASE)
ax.set_ylabel("R²  (test set)", fontweight="bold")
panel_label(ax, "(A)  Explained variance by modality", y=0.985)
ax.grid(axis="y", alpha=0.18)

for bar_, m_, s_ in zip(bars_, ab["R2_mean"], ab["R2_std"]):
    bar_top = bar_.get_height()
    bar_x   = bar_.get_x() + bar_.get_width() / 2
    
    if m_ > 0.20:
        y_pos = bar_top * 0.75
        va_str = "center"
    else:
        y_pos = bar_top + s_ + 0.015 
        va_str = "bottom"

    ax.text(bar_x, y_pos,
            f"{m_:.3f}\n±{s_:.3f}",
            ha="center", va=va_str,
            fontsize=FS_ANNOT - 0.5, fontweight="semibold",
            color="#222", linespacing=1.4,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, ec="none", zorder=10))

bars_[0].set_edgecolor("#222"); bars_[0].set_linewidth(2.0)
ax.axhline(0.81, color=C["red"], ls="--", lw=1.8, alpha=0.9,
           label="IHME sociodem. R²=0.81", zorder=8)
ax.legend(fontsize=FS_LEGEND-1, loc="upper left", framealpha=0.95,
          bbox_to_anchor=(0.01, 0.97))

# Panel B — MAE 
ax2 = axes[1]
bars2 = ax2.bar(x, ab["MAE_mean"], yerr=ab["MAE_std"], capsize=6,
                color=mae_cols, edgecolor="white", lw=0.9, alpha=0.98, zorder=3)
ylim_top_mae = (ab["MAE_mean"]+ab["MAE_std"]).max() * 1.25
ax2.set_ylim(0, ylim_top_mae)
ax2.set_xticks(x)
ax2.set_xticklabels(ab["label"], rotation=40, ha="right", fontsize=FS_BASE)
ax2.set_ylabel("MAE  (years)", fontweight="bold")
panel_label(ax2, "(B)  Mean Absolute Error by modality", y=0.985)
ax2.grid(axis="y", alpha=0.18)

for bar_, m_, s_ in zip(bars2, ab["MAE_mean"], ab["MAE_std"]):
    bar_top = bar_.get_height()
    bar_x   = bar_.get_x() + bar_.get_width() / 2
    inner_y = bar_top * 0.75 
    
    ax2.text(bar_x, inner_y,
             f"{m_:.2f}\n±{s_:.2f}",
             ha="center", va="center",
             fontsize=FS_ANNOT - 0.5, fontweight="semibold",
             color="#222", linespacing=1.4,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, ec="none"))

bars2[0].set_edgecolor("#222"); bars2[0].set_linewidth(2.0)

plt.tight_layout()
save_fig(fig, "fig07_master_summary_ablation_v5")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 08 — BRACKET BIAS + WATERFALLS (Split into A, B, C)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig08] Bracket bias + split waterfalls …")

t2       = tables["table2_le_bracket"]
brackets = list(t2.index)
maes_b   = t2["MAE"].values.astype(float)
rmses_b  = t2["RMSE"].values.astype(float)

if "Unique_Counties" in t2.columns:
    counts_b = t2["Unique_Counties"].values.astype(float)
elif "Count" in t2.columns:
    counts_b = t2["Count"].values.astype(float)
else:
    bracket_cnts = (predictions_df.groupby("LE_Bracket", observed=True)["fips"]
                    .nunique().reindex(brackets, fill_value=1).values.astype(float))
    counts_b = bracket_cnts

# --- Helper: build waterfall with forced non-overlapping text and legend ---
def build_waterfall(ax, example_row, tag, show_legend=True):
    # Pull the exact aligned row from the SHAP sample dataframe
    row_idx = int(example_row['Sample_Idx']) 
    shap_row = shap_values_arr[row_idx] 

    top10    = np.argsort(np.abs(shap_row))[::-1][:10]
    feat_lbl = [fn_list[i][:30]+("…" if len(fn_list[i])>30 else "") for i in top10]
    sv_v     = shap_row[top10]
    order_   = np.argsort(sv_v)
    sv_plot  = sv_v[order_]
    fl_plot  = [feat_lbl[i] for i in order_]

    y_wf = np.arange(len(sv_plot))
    c_wf = [C["green"] if v > 0 else C["red"] for v in sv_plot]
    bars_wf = ax.barh(y_wf, sv_plot, color=c_wf, alpha=0.85, height=0.65,
                      edgecolor="white", lw=0.8)
    
    # Safely expand x-limits so text outside bars never gets cropped
    max_val = float(np.abs(sv_plot).max())
    pad_text = max_val * 0.05
    ax.set_xlim(-max_val * 1.35, max_val * 1.35)

    # Always place text OUTSIDE the bar to prevent clipping or unreadable overlaps
    for bar_, val_ in zip(bars_wf, sv_plot):
        lbl = f"{val_:+.3f} yr"
        if val_ >= 0:
            ax.text(val_ + pad_text, bar_.get_y() + bar_.get_height() / 2, lbl, 
                    va="center", ha="left", fontsize=FS_ANNOT, color="#222", fontweight="semibold")
        else:
            ax.text(val_ - pad_text, bar_.get_y() + bar_.get_height() / 2, lbl, 
                    va="center", ha="right", fontsize=FS_ANNOT, color="#222", fontweight="semibold")
            
    ax.axvline(0, color="#444", lw=1.2)
    ax.set_yticks(y_wf)
    ax.set_yticklabels(fl_plot, fontsize=FS_LABEL - 1)
    ax.set_xlabel("SHAP value  (Δ LE prediction, years)", fontweight="bold")
    
    # ---> COMMENTED OUT: Redundant panel tags
    panel_label(ax, tag, y=0.985)
    
    # Increased padding (pad=20) for more open space at the top
    ax.set_title(f"Actual = {example_row['actual']:.1f} yr  |  "
                 f"Predicted = {example_row['predicted']:.1f} yr  |  "
                 f"Error = {example_row['abs_error']:.2f} yr",
                 fontweight="bold", loc="left", fontsize=FS_TITLE - 1, pad=20)
    ax.grid(axis="x", alpha=0.22)

    # Combined stats & legend forced BELOW the plot (y = -0.15) to prevent overlapping bottom bars
    net_shap = sv_plot.sum()
    inc = (sv_plot > 0).sum(); dec = (sv_plot <= 0).sum()
    combo_txt = (f"Net SHAP = {net_shap:+.3f} yr (top-10)\n"
                 f"↑ Increases LE: {inc} features\n"
                 f"↓ Decreases LE: {dec} features")
    
    ax.text(0.0, -0.15, combo_txt, transform=ax.transAxes,
            fontsize=FS_ANNOT, va="top", ha="left", zorder=15,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", alpha=0.95, ec="#AAAAAA", lw=0.8))

    if show_legend:
        pp = mpatches.Patch(color=C["green"], alpha=0.85, label="Increases LE")
        np_ = mpatches.Patch(color=C["red"],  alpha=0.85, label="Decreases LE")
        ax.legend(handles=[pp, np_], fontsize=FS_LEGEND - 1, framealpha=0.95,
                  loc="upper left", bbox_to_anchor=(0.4, -0.15))

# --- County Selection Logic ---
shap_df_w_metrics = predictions_df.loc[X_sample.index].copy()
shap_df_w_metrics['Sample_Idx'] = np.arange(len(shap_df_w_metrics))

median_county_shap_idx = np.argmin(np.abs(shap_df_w_metrics['actual'] - predictions_df['actual'].median()))
county_max_shap_idx = shap_df_w_metrics['actual'].idxmax()
county_min_shap_idx = shap_df_w_metrics['actual'].idxmin()

ex_good = shap_df_w_metrics.iloc[median_county_shap_idx]
ex_highLE = shap_df_w_metrics.loc[county_max_shap_idx]
ex_lowLE = shap_df_w_metrics.loc[county_min_shap_idx]

mid_le_mask = shap_df_w_metrics['LE_Bracket'] == "76.5-79" 
good_mid_le_counties_shap = shap_df_w_metrics[mid_le_mask].sort_values("abs_error")
if len(good_mid_le_counties_shap) > 10:
    ex_mode = good_mid_le_counties_shap.iloc[len(good_mid_le_counties_shap) // 2]
else:
    ex_mode = good_mid_le_counties_shap.iloc[0]

# ============================================================================
# --- FIG 08A: Bracket Bias ---
# ============================================================================
print("[fig08a] Bracket bias ...")
fig_8a, ax = plt.subplots(figsize=(10, 8))
# Bumped y to 1.05 for white space
fig_8a.suptitle("Prediction Honesty: Bracket-level bias audit", fontsize=FS_SUPTITLE, fontweight="bold", y=1.05)

y_pos = np.arange(len(brackets))
H = 0.35
bc_mae  = [C["red"], C["orange"], C["green"], C["green"], C["orange"]]
bc_rmse = [C["red"], C["orange"], C["sky"],   C["sky"],   C["orange"]]

ax.barh(y_pos + H/2, maes_b,  height=H, color=bc_mae, alpha=0.82, edgecolor="white", lw=0.9)
ax.barh(y_pos - H/2, rmses_b, height=H, color=bc_rmse, alpha=0.55, edgecolor="white", lw=0.9, hatch="///")

for i, (mae_, cnt_) in enumerate(zip(maes_b, counts_b)):
    ax.scatter(mae_ + 0.14, i, s=(cnt_ / counts_b.max()) * 380, color=bc_mae[i], alpha=0.30, edgecolors="none", zorder=2)
    ax.text(mae_ + 0.22, i, f"{int(cnt_):,}", va="center", fontsize=FS_ANNOT, color="#444")

ax.set_yticks(y_pos)
ax.set_yticklabels([f"LE = {b}" for b in brackets], fontsize=FS_LABEL)
ax.set_xlabel("Error (years)", fontweight="bold")

# ---> COMMENTED OUT: Redundant panel tags
# panel_label(ax, "(A) Solid = MAE · Hatched = RMSE · Bubble = count", y=0.985)

ax.grid(axis="x", alpha=0.22)
ax.axvline(1.0, color="#888", ls=":", lw=1.5)

mp = mpatches.Patch(color=C["green"], alpha=0.82, label="MAE  (solid)")
rp = mpatches.Patch(color=C["sky"],   alpha=0.55, hatch="///", label="RMSE  (hatched)")
bp = mlines.Line2D([0],[0], marker="o", color="w", mfc="#999", alpha=0.5, ms=10, label="Bubble = county count")
ax.legend(handles=[mp, rp, bp], fontsize=FS_LEGEND, framealpha=0.95, loc="upper right")
plt.tight_layout()
save_fig(fig_8a, "fig08a_bracket_bias_v5")

# ============================================================================
# --- FIG 08B: SHAP Waterfalls: Mode vs Median County ---
# ============================================================================
print("[fig08b] SHAP waterfalls (average) ...")
fig_8b, axes = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={"wspace": 0.60})
# Bumped y to 1.05 for white space
fig_8b.suptitle("SHAP Waterfalls: Average County Profiles", fontsize=FS_SUPTITLE, fontweight="bold", y=1.05)

# Notice we no longer care about passing the exact "(B1)..." string because it's commented out in the helper function
build_waterfall(axes[0], ex_mode,   "(A)")
build_waterfall(axes[1], ex_good,   "(B)")

plt.tight_layout()
save_fig(fig_8b, "fig08b_waterfall_mode_median_v5")

# ============================================================================
# --- FIG 08C: SHAP Waterfalls: Extreme Counties ---
# ============================================================================
print("[fig08c] SHAP waterfalls (extremes) ...")
fig_8c, axes = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={"wspace": 0.60})
# Bumped y to 1.05 for white space
fig_8c.suptitle("SHAP Waterfalls: Extreme County Profiles", fontsize=FS_SUPTITLE, fontweight="bold", y=1.05)

build_waterfall(axes[0], ex_highLE, "(A)")
build_waterfall(axes[1], ex_lowLE,  "(B)")

plt.tight_layout()
save_fig(fig_8c, "fig08c_waterfall_extremes_v5")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG 09 a, b, c — ENVIRONMENTAL LONGEVITY SIGNATURES ("The Cheatcode")
# Split into 3 standalone figures for perfect LaTeX/Overleaf integration.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[fig09] Environmental Longevity Signatures (Split into A, B, C) …")

try:
    # 1. --- Identify the Extremes ---
    county_le = predictions_df.groupby('fips')['actual'].mean().reset_index()
    
    p95_le = county_le['actual'].quantile(0.95)
    p05_le = county_le['actual'].quantile(0.05)
    
    county_le['Zone'] = 'National Average'
    county_le.loc[county_le['actual'] >= p95_le, 'Zone'] = 'Longevity Oasis (Top 5%)'
    county_le.loc[county_le['actual'] <= p05_le, 'Zone'] = 'Vulnerability Sink (Bottom 5%)'
    
    n_top = (county_le['Zone'] == 'Longevity Oasis (Top 5%)').sum()
    n_bot = (county_le['Zone'] == 'Vulnerability Sink (Bottom 5%)').sum()

    # 2. --- Aggregate Feature Data for the Profile ---
    KEY_PROFILE_FEATS = [
        "Nighttime Surface Temp (Mean)",
        "NDVI Mean",
        "Deciduous Forest %",
        "Developed (Med Intensity) %",
        "Soil Moisture Mean",
        "Corn %",
        "Elevation (10th Percentile)"
    ]
    
    valid_feats = [f for f in KEY_PROFILE_FEATS if f in df_full.columns]
    county_feats = df_full.groupby('fips')[valid_feats].mean().reset_index()
    
    prof_df = county_le[['fips', 'Zone', 'actual']].merge(county_feats, on='fips', how='inner')
    
    for feat in valid_feats:
        nat_mean = prof_df[feat].mean()
        nat_std  = prof_df[feat].std()
        prof_df[f"{feat}_z"] = (prof_df[feat] - nat_mean) / (nat_std + 1e-9)

    zone_profiles = prof_df.groupby('Zone')[[f"{f}_z" for f in valid_feats]].mean()

    # Palette
    COL_TOP = "#0072B2"  
    COL_BOT = "#D55E00"  
    COL_NAT = "#AAAAAA"  

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # FIG 09A: The Oasis & Sink Map
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    fig_9a, ax_map = plt.subplots(figsize=(14, 9))
    fig_9a.suptitle("Geographic Distribution of Environmental Longevity Extremes\n"
                    "Top 5% vs Bottom 5% Life Expectancy Counties (20-Year Average)",
                    fontsize=FS_SUPTITLE, fontweight="bold", y=0.98)

    counties_map = gpd.read_file(COUNTY_SHP)
    states_map   = gpd.read_file(STATE_SHP)
    counties_map = counties_map[counties_map["STATEFP"].isin(CONTINENTAL)]
    states_map   = states_map[states_map["STATEFP"].isin(CONTINENTAL)]
    
    geo_zones = counties_map.merge(county_le[['fips', 'Zone']], left_on="GEOID", right_on="fips", how="left")
    
    geo_zones.plot(ax=ax_map, color="#EAEAEA", edgecolor="white", lw=0.3)
    geo_zones[geo_zones['Zone'] == 'Longevity Oasis (Top 5%)'].plot(ax=ax_map, color=COL_TOP, edgecolor="none", zorder=3)
    geo_zones[geo_zones['Zone'] == 'Vulnerability Sink (Bottom 5%)'].plot(ax=ax_map, color=COL_BOT, edgecolor="none", zorder=3)
    states_map.boundary.plot(ax=ax_map, lw=0.6, edgecolor="white", alpha=0.9, zorder=4)
    
    bounds = states_map.total_bounds
    ax_map.set_xlim(bounds[0] - 1, bounds[2] + 1)
    ax_map.set_ylim(bounds[1] - 1, bounds[3] + 1)
    ax_map.axis("off")
    
    panel_label(ax_map, "", x=0.01, y=0.99)
    
    leg_top = mpatches.Patch(color=COL_TOP, label=f"Longevity Oases (Top 5%, n={n_top})\nAvg LE ≥ {p95_le:.1f} yrs")
    leg_bot = mpatches.Patch(color=COL_BOT, label=f"Vulnerability Sinks (Bottom 5%, n={n_bot})\nAvg LE ≤ {p05_le:.1f} yrs")
    # Legend moved cleanly into the Pacific Ocean space
    ax_map.legend(handles=[leg_top, leg_bot], loc="lower left", fontsize=FS_LEGEND+1, framealpha=0.95, bbox_to_anchor=(0.02, 0.05))

    plt.tight_layout()
    save_fig(fig_9a, "fig09a_longevity_map_v5")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # FIG 09B: Temporal Trajectory
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    fig_9b, ax_traj = plt.subplots(figsize=(10, 6.5))
    fig_9b.suptitle("20-Year Structural Divergence in Life Expectancy", 
                    fontsize=FS_SUPTITLE, fontweight="bold", y=0.98)

    pred_merged = predictions_df.merge(county_le[['fips', 'Zone']], on='fips', how='left')
    yearly_trends = pred_merged.groupby(['year', 'Zone'])['actual'].mean().unstack()
    years_arr = yearly_trends.index.values

    ax_traj.plot(years_arr, yearly_trends['Longevity Oasis (Top 5%)'], "o-", color=COL_TOP, lw=3, ms=8, label="Top 5% Counties")
    ax_traj.plot(years_arr, yearly_trends['Vulnerability Sink (Bottom 5%)'], "o-", color=COL_BOT, lw=3, ms=8, label="Bottom 5% Counties")
    ax_traj.plot(years_arr, yearly_trends['National Average'], "--", color=COL_NAT, lw=2.5, label="National Average")
    
    for zone, col in zip(['Longevity Oasis (Top 5%)', 'Vulnerability Sink (Bottom 5%)'], [COL_TOP, COL_BOT]):
        zf = np.polyfit(years_arr, yearly_trends[zone], 1)
        ax_traj.plot(years_arr, np.poly1d(zf)(years_arr), ":", color=col, lw=2, alpha=0.7)

    ax_traj.set_ylabel("Actual Life Expectancy (Years)", fontweight="bold", fontsize=FS_LABEL)
    ax_traj.set_xlabel("Year", fontweight="bold", fontsize=FS_LABEL)
    ax_traj.xaxis.set_major_locator(MultipleLocator(4))
    ax_traj.grid(True, alpha=0.25)
    
    # Legend moved INSIDE to the center-left (fits perfectly between the lines)
    ax_traj.legend(fontsize=FS_LEGEND, loc="center left", framealpha=0.95)
    panel_label(ax_traj, "", x=0.02, y=0.99)
    
    gap_2000 = yearly_trends['Longevity Oasis (Top 5%)'].iloc[0] - yearly_trends['Vulnerability Sink (Bottom 5%)'].iloc[0]
    gap_2019 = yearly_trends['Longevity Oasis (Top 5%)'].iloc[-1] - yearly_trends['Vulnerability Sink (Bottom 5%)'].iloc[-1]
    stat_box(ax_traj, f"The Longevity Gap:\n{gap_2000:.1f} yrs (2000) → {gap_2019:.1f} yrs (2019)", loc="upper left")

    ax_traj.spines["top"].set_visible(False)
    ax_traj.spines["right"].set_visible(False)

    plt.tight_layout()
    save_fig(fig_9b, "fig09b_longevity_trajectory_v5")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # FIG 09C: The "Cheatcode" Profile (Z-Score Diverging Bar)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    fig_9c, ax_bar = plt.subplots(figsize=(10, 8))
    fig_9c.suptitle("The Environmental Blueprint (Z-Score Profile)\n"
                    "Feature divergence measured in Standard Deviations from National Average", 
                    fontsize=FS_SUPTITLE, fontweight="bold", y=0.98)

    y_pos = np.arange(len(valid_feats))
    height = 0.35
    
    top_z = zone_profiles.loc['Longevity Oasis (Top 5%)'].values
    bot_z = zone_profiles.loc['Vulnerability Sink (Bottom 5%)'].values
    
    clean_labels = [f.replace(" (Mean)","").replace(" (10th Percentile)","").replace(" %", " Coverage") for f in valid_feats]

    ax_bar.barh(y_pos + height/2, top_z, height, color=COL_TOP, label="Longevity Oases")
    ax_bar.barh(y_pos - height/2, bot_z, height, color=COL_BOT, label="Vulnerability Sinks")
    
    ax_bar.axvline(0, color="black", lw=1.2)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(clean_labels, fontweight="bold", fontsize=FS_LABEL)
    ax_bar.set_xlabel("Standard Deviations from National Avg (Z-Score)", fontweight="bold", fontsize=FS_LABEL)
    ax_bar.grid(axis="x", alpha=0.25)
    
    max_z_val = max(np.abs(top_z).max(), np.abs(bot_z).max())
    ax_bar.set_xlim(-max_z_val * 1.3, max_z_val * 1.3)
    
    for i, (tz, bz) in enumerate(zip(top_z, bot_z)):
        ha_top = 'left' if tz > 0 else 'right'
        offset_top = 0.05 if tz > 0 else -0.05
        ax_bar.text(tz + offset_top, i + height/2, f"{tz:+.1f} SD", 
                    va='center', ha=ha_top, color=COL_TOP, fontsize=FS_ANNOT+1, fontweight="bold")
        
        ha_bot = 'left' if bz > 0 else 'right'
        offset_bot = 0.05 if bz > 0 else -0.05
        ax_bar.text(bz + offset_bot, i - height/2, f"{bz:+.1f} SD", 
                    va='center', ha=ha_bot, color=COL_BOT, fontsize=FS_ANNOT+1, fontweight="bold")

    # Legend moved INSIDE to the upper left (Interpretation box removed)
    ax_bar.legend(fontsize=FS_LEGEND, loc="upper left", framealpha=0.95)
    panel_label(ax_bar, "", x=0.02, y=0.99)

    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    plt.tight_layout()
    save_fig(fig_9c, "fig09c_longevity_profile_v5")

except Exception as e:
    print(f"  ⚠ fig09 error: {e}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG A1 — NIGHTTIME THERMAL PARADOX: SUMMARY (bar + cumulative + channel totals)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[figA1] Nighttime thermal paradox (summary) …")

# EXACT same masks as your verification script → always matches 4.69× / 4.73×
night_mask = shap_importance["Feature"].str.contains(
    "Nighttime|Cooling Efficiency|Night ", case=False, na=False)
day_mask   = shap_importance["Feature"].str.contains(
    "Daytime|Day ", case=False, na=False)

night_p = shap_importance.loc[night_mask, "Feature"].tolist()
day_p   = shap_importance.loc[day_mask,   "Feature"].tolist()

def mean_abs_shap(feat):
    if feat not in fn_list: return 0.0
    idx = fn_list.index(feat)
    return float(np.mean(np.abs(shap_values_arr[:, idx])))

ns = np.array([mean_abs_shap(f) for f in night_p])
ds = np.array([mean_abs_shap(f) for f in day_p])
nl = [f.replace("Nighttime ","").replace(" (","\n(") for f in night_p]
dl = [f.replace("Daytime ",  "").replace(" (","\n(") for f in day_p]

tn = ns.sum()
td = ds.sum()
ratio = tn / max(td, 1e-6)

fig_a1 = plt.figure(figsize=(18, 9))
gs_a1  = gridspec.GridSpec(1, 3, figure=fig_a1, hspace=0.40, wspace=0.38,
                           width_ratios=[1.8, 1.2, 1.0])
ax_bar = fig_a1.add_subplot(gs_a1[0, 0])
ax_cum = fig_a1.add_subplot(gs_a1[0, 1])
ax_tot = fig_a1.add_subplot(gs_a1[0, 2])

fig_a1.suptitle(
    "The Nighttime Thermal Paradox: Minimum Cooling Opportunity\n"
    f"Outperforms Peak Daytime Heat as a Predictor of County-Level Longevity\n"
    f"(Night/Day |SHAP| ratio = {ratio:.2f}×)",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.02)

# A1 – ranked bar
all_n = night_p + day_p
all_s = list(ns) + list(ds)
all_l = nl + dl
all_c = [C["night"]]*len(night_p) + [C["day"]]*len(day_p)
ord_  = np.argsort(all_s)
bars_a = ax_bar.barh(np.arange(len(ord_)), [all_s[i] for i in ord_],
                     color=[all_c[i] for i in ord_], alpha=0.88, height=0.65,
                     edgecolor="white", lw=0.8)
ax_bar.set_yticks(np.arange(len(ord_)))
ax_bar.set_yticklabels([all_l[i] for i in ord_], fontsize=FS_BASE-0.5)
ax_bar.set_xlabel("Mean |SHAP| value  (years)", fontweight="bold")
panel_label(ax_bar, "(A)  All LST features ranked by SHAP importance", y=0.985)

leg_n = mpatches.Patch(color=C["night"], label="Nighttime")
leg_d = mpatches.Patch(color=C["day"], label="Daytime")
ax_bar.legend(handles=[leg_n, leg_d], loc="lower left", bbox_to_anchor=(0.0, 1.01),
              ncol=2, frameon=False, fontsize=FS_TITLE, handlelength=0.7, handletextpad=0.4)

ax_bar.grid(axis="x", alpha=0.2)
for bar_, val_ in zip(bars_a, [all_s[i] for i in ord_]):
    ax_bar.text(val_+0.001, bar_.get_y()+bar_.get_height()/2,
                f"{val_:.3f}", va="center", fontsize=FS_ANNOT, fontweight="bold")

stat_box(ax_bar,
         f"Σ|SHAP| Night = {tn:.3f} yr   Night (n={len(night_p)})\n"
         f"Σ|SHAP| Day   = {td:.3f} yr   Day   (n={len(day_p)})\n"
         f"Night/Day ratio = {ratio:.2f}×",
         loc="lower right", fs=FS_ANNOT, fc="#EEF0FF")

# A2 – cumulative
ns_s = np.sort(ns)[::-1]
ds_s = np.sort(ds)[::-1]
ax_cum.step(np.arange(1,len(ns_s)+1), np.cumsum(ns_s), color=C["night"],
            lw=2.5, where="post", label="Nighttime")
ax_cum.step(np.arange(1,len(ds_s)+1), np.cumsum(ds_s), color=C["day"],
            lw=2.5, where="post", label="Daytime", ls="--")
ax_cum.fill_between(np.arange(1,len(ns_s)+1), 0, np.cumsum(ns_s),
                    color=C["night"], alpha=0.15, step="post")
ax_cum.fill_between(np.arange(1,len(ds_s)+1), 0, np.cumsum(ds_s),
                    color=C["day"], alpha=0.12, step="post")
ax_cum.set_xlabel("Features included (ranked)", fontweight="bold", fontsize=FS_LABEL)
ax_cum.set_ylabel("Cumulative mean |SHAP|  (years)", fontweight="bold")
panel_label(ax_cum, "(B)  Cumulative contribution", y=0.985)
ax_cum.legend(fontsize=FS_LEGEND, framealpha=0.95, loc="lower right")
ax_cum.grid(True, alpha=0.2)

# A3 – total bar
ax_tot.bar(["Nighttime\nLST","Daytime\nLST"], [tn, td],
           color=[C["night"],C["day"]], alpha=0.88, width=0.5,
           edgecolor="white", lw=1.2)
for xp, val_ in enumerate([tn,td]):
    ax_tot.text(xp, val_ + max(tn,td)*0.025,
                f"{val_:.3f} yr",
                ha="center", va="bottom", fontsize=FS_LABEL, fontweight="bold")
ax_tot.set_ylabel("Total mean |SHAP|  (years)", fontweight="bold")
panel_label(ax_tot, "(C)  Channel-level total", y=0.985)
ax_tot.grid(axis="y", alpha=0.2)
ax_tot.set_ylim(0, max(tn, td)*1.35)
ax_tot.spines["top"].set_visible(False)
ax_tot.spines["right"].set_visible(False)

ax_tot.hlines(y=td, xmin=0, xmax=1, color=C["day"], linestyle="--", lw=1.5, alpha=0.8)
ax_tot.annotate("", xy=(0.35, tn), xytext=(0.35, td),
                arrowprops=dict(arrowstyle="<->", color=C["night"], lw=2.5))
ax_tot.text(0.42, (tn + td) / 2,
            f"{ratio:.2f}×\nImpact", ha="left", va="center",
            fontsize=FS_TITLE, color=C["night"], fontweight="bold")
ax_tot.text(0.95, 0.88,
            f"Nighttime LST impact is\n{ratio*100:.0f}% of Daytime LST impact",
            ha="right", va="top", transform=ax_tot.transAxes,
            fontsize=FS_ANNOT, color=C["night"], fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.5", fc="#EEF0FF",
                      alpha=0.95, ec=C["night"], lw=1.2))

plt.tight_layout()
save_fig(fig_a1, "figA1_nighttime_paradox_summary_v5")
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG A2 — NIGHTTIME THERMAL PARADOX: LST DEPENDENCE DEEP-DIVE  (split from A)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[figA2] Nighttime thermal paradox (dependence) …")

fig_a2 = plt.figure(figsize=(16, 7))
gs_a2  = gridspec.GridSpec(1, 2, figure=fig_a2, wspace=0.36)
ax_dep1 = fig_a2.add_subplot(gs_a2[0, 0])
ax_dep2 = fig_a2.add_subplot(gs_a2[0, 1])

fig_a2.suptitle(
    "Nighttime vs Daytime LST: SHAP Dependence Profiles\n"
    f"Nighttime thermal cooling provides a much stronger, earlier physiological threshold (ratio {ratio:.2f}×)",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.02)

def shap_dep_panel(ax, feat, color, tag, title_str):
    if feat not in fn_list:
        ax.text(0.5,0.5,f"'{feat}' not found",
                ha="center",va="center",transform=ax.transAxes); return
    fi = fn_list.index(feat)
    xv_ = X_sample[feat].values.copy()
    yv_ = shap_values_arr[:,fi].copy()
    sc = ax.scatter(xv_, yv_, c=xv_, cmap="coolwarm", alpha=0.30, s=15,
               linewidths=0, rasterized=True,
               vmin=np.nanpercentile(xv_, 2), vmax=np.nanpercentile(xv_, 98))
    fig_a2.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label="Feature value (°C)")
    lxd, lyd = lowess_trend(xv_, yv_)
    ax.plot(lxd, lyd, color=color, lw=3.0, zorder=5)
    ax.axhline(0, color="#888", lw=2.0, ls="--")
    sign_changes = np.where(np.diff(np.sign(lyd)))[0]
    x_range = float(xv_.max() - xv_.min())
    for ki, sci in enumerate(sign_changes[:1]):   # show first threshold
        xc_ = float(lxd[sci])
        ax.axvline(xc_, color=color, lw=1.5, ls=":", alpha=0.8)
        ax.text(xc_ + x_range*0.03,
                float(np.nanpercentile(yv_, 94)),
                f"Threshold\n≈{xc_:.1f}°C",
                fontsize=FS_ANNOT, color=color, ha="left", va="top",
                bbox=dict(fc="white", alpha=0.92, ec=color, lw=0.8))
    ax.set_xlabel(f"{feat}  (°C)", fontweight="bold", fontsize=FS_LABEL)
    ax.set_ylabel("SHAP value  (years)", fontweight="bold")
    panel_label(ax, tag, y=0.985)
    ax.set_title(title_str, fontweight="bold", loc="left",
                 fontsize=FS_TITLE, pad=8)
    ax.grid(True, alpha=0.2)
    stat_box(ax, f"Mean |SHAP| = {float(np.mean(np.abs(yv_))):.3f} yr",
             loc="lower right", fs=FS_ANNOT)

shap_dep_panel(ax_dep1,
               "Nighttime Temp (10th Percentile)", C["night"],
               "(A)  10th %ile Nighttime LST",
               "Nighttime: strong protective effect below threshold")
shap_dep_panel(ax_dep2,
               "Daytime Temp (10th Percentile)", C["day"],
               "(B)  10th %ile Daytime LST",
               "Daytime: weaker, delayed harm threshold")

plt.tight_layout()
save_fig(fig_a2, "figA2_nighttime_dependence_v5")

# Fig B is now a table in the main text, so skipping the old Fig B code and going straight to Fig C.

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG C — SPATIAL PREDICTION FIDELITY (With marginal densities)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[figC] Spatial prediction fidelity …")
try:
    counties = gpd.read_file(COUNTY_SHP)
    states   = gpd.read_file(STATE_SHP)
    counties = counties[counties["STATEFP"].isin(CONTINENTAL)]
    states   = states[states["STATEFP"].isin(CONTINENTAL)]

    yr_cnt   = predictions_df.groupby("year")["fips"].count()
    rep_yr   = int(yr_cnt.idxmax())
    pred_yr  = predictions_df[predictions_df["year"]==rep_yr].copy()
    pred_yr["fips"] = pred_yr["fips"].astype(str).str.zfill(5)

    geo_c = counties.merge(pred_yr[["fips","actual","predicted"]],
                           left_on="GEOID", right_on="fips", how="left")

    le_lo = float(np.nanpercentile(pred_yr[["actual","predicted"]].values, 2))
    le_hi = float(np.nanpercentile(pred_yr[["actual","predicted"]].values, 98))
    norm_le = mcolors.Normalize(vmin=le_lo, vmax=le_hi)
    sm_le   = cm.ScalarMappable(cmap=CMAP_LE, norm=norm_le)

    fig_c = plt.figure(figsize=(22, 12))
    gs_c  = gridspec.GridSpec(2, 2, figure=fig_c,
                               width_ratios=[2.4, 1.1],
                               height_ratios=[1.0, 1.0],
                               hspace=0.28, wspace=0.2)
    ax_act  = fig_c.add_subplot(gs_c[0,0])
    ax_pred = fig_c.add_subplot(gs_c[1,0])
    ax_sct  = fig_c.add_subplot(gs_c[0,1])
    ax_hst  = fig_c.add_subplot(gs_c[1,1])

    # --- FIX 1: Bumped up to 1.05 to create open space for titles ---
    fig_c.suptitle(
        f"Spatial Prediction Fidelity: Actual vs Predicted Life Expectancy ({rep_yr})\n"
        "Side-by-side choropleth on identical scale demonstrates geographic pattern replication",
        fontsize=FS_SUPTITLE, fontweight="bold", y=1.05)

    def plot_le_map(ax_, col_, tag_, lbl_):
        geo_c.plot(column=col_, cmap=CMAP_LE, ax=ax_,
                   vmin=le_lo, vmax=le_hi,
                   missing_kwds=dict(color=C["lgrey"]), legend=False)
        states.boundary.plot(ax=ax_, lw=0.5, edgecolor="white", alpha=0.8)
        cb_ = fig_c.colorbar(sm_le, ax=ax_, orientation="vertical",
                              fraction=0.014, pad=0.02, shrink=0.85)
        cb_.set_label(lbl_, fontsize=FS_ANNOT)
        cb_.ax.tick_params(labelsize=FS_ANNOT)
        
        # --- FIX 2: Swapped panel_label for an outside set_title ---
        ax_.set_title(tag_, loc="left", fontweight="bold", fontsize=FS_TITLE, pad=12)
        
        vals_ = geo_c[col_].dropna()
        stat_box(ax_,
                 f"Mean = {vals_.mean():.2f} yr\n"
                 f"Range = {vals_.min():.1f}–{vals_.max():.1f} yr",
                 loc="lower left", fs=FS_ANNOT)
        ax_.axis("off")

    # --- FIX 3: Renamed C1 and C2 to A and B ---
    plot_le_map(ax_act,  "actual",    f"(A)  Actual LE — {rep_yr}  (IHME)",
                "Actual LE (years)")
    plot_le_map(ax_pred, "predicted", f"(B)  Predicted LE — {rep_yr}  (Random Forest)",
                "Predicted LE (years)")

    # --- Panel C3: Scatter actual vs predicted with Marginal Density ---
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    av  = pred_yr["actual"].dropna().values
    pv  = pred_yr["predicted"].dropna().values
    mk  = ~(np.isnan(av)|np.isnan(pv))
    av  = av[mk]; pv = pv[mk]
    r2_ = 1 - np.sum((av-pv)**2)/np.sum((av-av.mean())**2)
    mae_= np.mean(np.abs(av-pv))
    rr, _ = stats.pearsonr(av, pv)
    ae  = np.abs(av-pv)

    # Main Scatter
    sc_ = ax_sct.scatter(av, pv, c=ae, cmap=CMAP_ERR, alpha=0.45, s=15, linewidths=0, rasterized=True, vmin=0, vmax=np.percentile(ae,95))
    
    lo_, hi_ = min(av.min(),pv.min())-0.5, max(av.max(),pv.max())+0.5
    ax_sct.plot([lo_,hi_],[lo_,hi_],"k--",lw=1.5,alpha=0.6,label="1:1 line")
    sl_, ic_, _, _, _ = stats.linregress(av, pv)
    xl_ = np.linspace(lo_,hi_,200)
    ax_sct.plot(xl_, sl_*xl_+ic_, color=C["blue"], lw=2, alpha=0.8, label=f"OLS (slope={sl_:.3f})")
    
    ax_sct.set_xlim(lo_,hi_); ax_sct.set_ylim(lo_,hi_)
    ax_sct.set_xlabel(f"Actual LE (years, IHME {rep_yr})", fontweight="bold")
    ax_sct.set_ylabel("Predicted LE (years)", fontweight="bold")
    
    # Removed old internal panel_label
    # panel_label(ax_sct, "(C3)  Scatter: actual vs predicted", y=0.985)
    
    ax_sct.legend(fontsize=FS_LEGEND-1, loc="upper left", framealpha=0.95)
    ax_sct.grid(True, alpha=0.2)
    ax_sct.set_aspect("equal")
    
    # Append marginal axes to the scatter plot
    divider = make_axes_locatable(ax_sct)
    ax_histx = divider.append_axes("top", 1.2, pad=0.1, sharex=ax_sct)
    ax_histy = divider.append_axes("right", 1.2, pad=0.1, sharey=ax_sct)

    # Calculate and plot KDE curves
    x_grid = np.linspace(lo_, hi_, 200)
    kde_av = stats.gaussian_kde(av)
    kde_pv = stats.gaussian_kde(pv)
    
    ax_histx.plot(x_grid, kde_av(x_grid), color=C["blue"], lw=2, label="Actual Dist.")
    ax_histx.fill_between(x_grid, 0, kde_av(x_grid), color=C["blue"], alpha=0.2)
    
    ax_histy.plot(kde_pv(x_grid), x_grid, color=C["orange"], lw=2, label="Pred. Dist.")
    ax_histy.fill_betweenx(x_grid, 0, kde_pv(x_grid), color=C["orange"], alpha=0.2)

    # Clean up marginal axes
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.tick_params(axis="y", labelleft=False)
    ax_histx.axis("off"); ax_histy.axis("off")
    
    # --- FIX 4: Place the (C) title on the top marginal axis so it doesn't overlap the histograms ---
    ax_histx.set_title("(C)  Scatter: actual vs predicted", loc="left", fontweight="bold", fontsize=FS_TITLE, pad=12)

    # Add Colorbar nicely bounded below the margins
    cb_ = fig_c.colorbar(sc_, ax=ax_sct, fraction=0.046, pad=0.04)
    cb_.set_label("Absolute error (years)", fontsize=FS_ANNOT)
    cb_.ax.tick_params(labelsize=FS_ANNOT)

    stat_box(ax_sct,
             f"R² = {r2_:.3f}\nMAE = {mae_:.2f} yr\nr = {rr:.3f}\n"
             f"OLS slope = {sl_:.3f}\nn = {len(av):,}",
             loc="lower right", fs=FS_ANNOT, fc="#E8F5E9")

    # --- Panel C4: Histogram ---
    resid_ = av - pv
    ax_hst.hist(resid_, bins=60, color=C["blue"], alpha=0.75,
                edgecolor="white", lw=0.3, density=True, label="Residuals")
    mu_, sd_ = resid_.mean(), resid_.std()
    xn_ = np.linspace(resid_.min(), resid_.max(), 300)
    ax_hst.plot(xn_, stats.norm.pdf(xn_,mu_,sd_), color=C["red"], lw=2,
                label=f"N({mu_:.3f}, {sd_:.2f}²)")
    ax_hst.axvline(0,   color="#444", lw=1.2, ls="--", alpha=0.7)
    ax_hst.axvline(mu_, color=C["red"], lw=1.5, ls=":", alpha=0.8,
                   label=f"Mean = {mu_:.3f} yr")
    ax_hst.axvspan(-1, 1, color=C["green"], alpha=0.10, label="±1 yr window")
    w1_ = (np.abs(resid_)<=1).mean()*100
    w2_ = (np.abs(resid_)<=2).mean()*100
    ax_hst.set_xlabel("Residual  (Actual−Predicted, years)", fontweight="bold")
    ax_hst.set_ylabel("Probability density", fontweight="bold")
    
    # --- FIX 5: Moved (D) title outside the plot ---
    ax_hst.set_title("(D)  Residual distribution", loc="left", fontweight="bold", fontsize=FS_TITLE, pad=12)
    # panel_label(ax_hst, "(C4)  Residual distribution", y=0.985)
    
    ax_hst.legend(fontsize=FS_LEGEND-1, framealpha=0.95, loc="upper left")
    ax_hst.grid(True, alpha=0.2)
    stat_box(ax_hst,
             f"Within ±1 yr: {w1_:.0f}%\nWithin ±2 yr: {w2_:.0f}%\n"
             f"Skewness = {stats.skew(resid_):.2f}\n"
             f"N({mu_:.3f}, {sd_:.2f}²)",
             loc="upper right", fs=FS_ANNOT)

    save_fig(fig_c, "figC_spatial_fidelity_v5")
except Exception as e:
    print(f"  ⚠ figC error: {e}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIG D — NEW: CROSS-MODAL SHAP INTERACTION MATRIX
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("[figD] Cross-modal SHAP interaction matrix (NEW) …")

# Select top 12 features across diverse modalities
D_FEATS_NAMES = [
    "Nighttime Cooling Efficiency",
    "Nighttime Temp (10th Percentile)",
    "Nighttime Temp (25th Percentile)",
    "Developed (Med Intensity) %",
    "Nighttime Surface Temp (Mean)",
    "Impervious Surface Heat Index",
    "Thermal Vegetation Index (TVI)",
    "Horse Density",
    "Elevation (10th Percentile)",
    "Wetland Flood Risk Index",
    "Daytime Temp (25th Percentile)",
    "Soil Moisture Variance",
]
# Filter to those actually present in SHAP array
D_FEATS = [f for f in D_FEATS_NAMES if f in fn_list][:12]
D_IDXS  = [fn_list.index(f) for f in D_FEATS]
SHAP_D  = shap_values_arr[:, D_IDXS]   # shape (n_samples, n_feats)

# Short display labels
short_labels = []
for f in D_FEATS:
    lab = (f.replace("Nighttime ", "Night ")
            .replace("Daytime ", "Day ")
            .replace(" (%)", "")
            .replace("Temp (", "T(")
            .replace("Percentile)", "pct)")
            .replace("Surface ", "Surf. ")
            .replace("Elevation", "Elev.")
            .replace("Moisture", "Moist.")
            .replace("Intensity", "Int."))
    short_labels.append(lab[:22])

# Build correlation matrix of SHAP absolute values
abs_shap = np.abs(SHAP_D)
corr_mat = np.corrcoef(abs_shap.T)   # (n_feats × n_feats)

# Mean |SHAP| per feature (for diagonal / bubble sizes)
mean_abs = np.mean(abs_shap, axis=0)

# Category colours per feature
D_CATS  = [categorize(f) for f in D_FEATS]
D_CCOLS = [CAT_PAL[c] for c in D_CATS]

n_feat = len(D_FEATS)

fig_d = plt.figure(figsize=(16, 13))
gs_d  = gridspec.GridSpec(1, 2, figure=fig_d,
                           width_ratios=[1.8, 0.7],
                           wspace=0.7)
ax_heat = fig_d.add_subplot(gs_d[0, 0])
ax_bar_d= fig_d.add_subplot(gs_d[0, 1])

fig_d.suptitle(
    "Cross-Modal SHAP Interaction Structure\n"
    "Correlation of feature importance patterns reveals synergistic & antagonistic modality pairs",
    fontsize=FS_SUPTITLE, fontweight="bold", y=1.02)

# Heatmap of SHAP correlation
cmap_corr = sns.diverging_palette(220, 20, as_cmap=True)
im = ax_heat.imshow(corr_mat, cmap=cmap_corr, vmin=-1, vmax=1, aspect="auto")
# Colorbar with extra padding so it doesn't overlap D2 tick labels
cbar = fig_d.colorbar(im, ax=ax_heat, fraction=0.035, pad=0.08)
cbar.set_label("Pearson r  (|SHAP_i| vs |SHAP_j|)", fontsize=FS_ANNOT)
cbar.ax.tick_params(labelsize=FS_ANNOT)

# Annotate each cell
for i in range(n_feat):
    for j in range(n_feat):
        val = corr_mat[i, j]
        txt_col = "white" if abs(val) > 0.55 else "#222"
        ax_heat.text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=7, color=txt_col, fontweight="bold" if i==j else "normal")

# Axis labels with category colour
ax_heat.set_xticks(range(n_feat))
ax_heat.set_yticks(range(n_feat))
ax_heat.set_xticklabels(short_labels, rotation=45, ha="right",
                         fontsize=FS_BASE-1.5)
ax_heat.set_yticklabels(short_labels, fontsize=FS_BASE-1.5)
for tick, col in zip(ax_heat.get_xticklabels(), D_CCOLS):
    tick.set_color(col); tick.set_fontweight("semibold")
for tick, col in zip(ax_heat.get_yticklabels(), D_CCOLS):
    tick.set_color(col); tick.set_fontweight("semibold")

# ── D1 subtitle ABOVE the heatmap (not inside it) ──
ax_heat.set_title("(A)  |SHAP| correlation matrix  (tick colour = modality)",
                  fontweight="bold", loc="left", fontsize=FS_TITLE, pad=10)

# Add a modality legend below the heatmap
cat_present = list(dict.fromkeys(D_CATS))
leg_d = [mpatches.Patch(color=CAT_PAL[c], label=c, alpha=0.85)
         for c in cat_present]
ax_heat.legend(handles=leg_d, title="Modality", fontsize=FS_LEGEND-1,
               title_fontsize=FS_LEGEND-1, framealpha=0.95,
               loc="upper right", bbox_to_anchor=(1.0, -0.20),
               ncol=3)

# Compute r threshold dynamically: median of off-diagonal nighttime correlations
night_idxs = [i for i, f in enumerate(D_FEATS) if "nighttime" in f.lower() or "night" in f.lower()]
if len(night_idxs) >= 2:
    off_diag_night = [corr_mat[i, j] for i in night_idxs for j in night_idxs if i != j]
    r_thresh = round(float(np.median(off_diag_night)), 1)
else:
    r_thresh = 0.7
# # Place annotation BELOW the heatmap axes (not on top of data)
# fig_d.text(0.35, -0.03,
#            f"Nighttime LST features form a strongly correlated cluster (r>{r_thresh:.1f})\n"
#            "— indicative of a unified thermal cooling mechanism.",
#            ha="center", fontsize=FS_ANNOT, style="italic", color=C["night"],
#            bbox=dict(boxstyle="round,pad=0.45", fc="#EEF0FF", alpha=0.9,
#                      ec=C["night"], lw=0.8))

# Right bar: mean |SHAP| ranked — with its own title ABOVE
ord_d   = np.argsort(mean_abs)
cols_d  = [D_CCOLS[i] for i in ord_d]
bars_d  = ax_bar_d.barh(np.arange(n_feat),
                         mean_abs[ord_d],
                         color=cols_d, alpha=0.88,
                         height=0.65, edgecolor="white", lw=0.8)
ax_bar_d.set_yticks(np.arange(n_feat))
ax_bar_d.set_yticklabels([short_labels[i] for i in ord_d], fontsize=FS_BASE-1)
for tick, ci in zip(ax_bar_d.get_yticklabels(), ord_d):
    tick.set_color(D_CCOLS[ci]); tick.set_fontweight("semibold")
ax_bar_d.set_xlabel("Mean |SHAP|  (years)", fontweight="bold")
# ── D2 title ABOVE the bar chart with spacing from colorbar ──
ax_bar_d.set_title("(B)  Individual mean |SHAP|",
                   fontweight="bold", loc="left", fontsize=FS_TITLE, pad=10)
ax_bar_d.grid(axis="x", alpha=0.2)
for bar_, val_ in zip(bars_d, mean_abs[ord_d]):
    ax_bar_d.text(val_+0.001, bar_.get_y()+bar_.get_height()/2,
                  f"{val_:.3f}", va="center", fontsize=FS_ANNOT, fontweight="bold")

plt.tight_layout()
save_fig(fig_d, "figD_cross_modal_interaction_v5")


# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print(f"  All figures (v5) → {OUT.absolute()}")
print("="*65)


Loading results …
  Data loaded successfully.

[fig01a/b] Spatial residual maps …
  ✓  fig01a_spatial_mae_v5.pdf / .png
  ✓  fig01b_spatial_residual_v5.pdf / .png
[fig02] Temporal combined panel …
  ✓  fig02_temporal_combined_v5.pdf / .png
[fig03] SHAP beeswarm …
  ✓  fig03_shap_beeswarm_v5.pdf / .png
[fig04a] Metabolic breakpoints — Inflection vs Threshold …
  ✓  fig04a_metabolic_breakpoints_v5.pdf / .png
[fig04b] Physics coupling …
⚠️ WARNING: Could not find raw feature matching Thermal Vegetation Index (TVI) in df_full
  ✓  fig04b_physics_coupling_v5.pdf / .png
[fig05] Forest buffer …
  ✓  fig05_forest_buffer_v5.pdf / .png
[fig06] Soil gradient …
  ✓  fig06_soil_gradient_v5.pdf / .png
[fig07] Ablation / master summary …
  ✓  fig07_master_summary_ablation_v5.pdf / .png
[fig08] Bracket bias + split waterfalls …
[fig08a] Bracket bias ...
  ✓  fig08a_bracket_bias_v5.pdf / .png
[fig08b] SHAP waterfalls (average) ...
  ✓  fig08b_waterfall_mode_median_v5.pdf / .png
[fig08c] SHAP waterfalls